# Fight and Agitation Detection: Pose-Based Motion Classification

This notebook builds and evaluates a per-person motion classifier for detecting aggressive behavior from CCTV footage, with the eventual target application being agitation and aggression detection in dementia and Parkinson's patients in care settings.

## What the model does

The system does not classify raw video frames directly. Instead, it works in three stages:

1. **Pose extraction.** For every frame, YOLOv8-pose detects each person and estimates 17 skeletal keypoints (nose, shoulders, elbows, wrists, hips, knees, ankles). ByteTrack assigns a consistent identity to each person across frames, so the same individual's keypoints can be followed over time.
2. **Normalization.** Each frame's keypoints are centered on the hip midpoint and scaled by shoulder width, so the model learns motion shape rather than raw pixel position, camera distance, or frame location.
3. **Classification.** Each tracked person's keypoint sequence (variable length, padded and masked) is passed through an LSTM that classifies the sequence as aggressive or calm.

The classifier operates **per person, not per pair**. Most existing fight detection work assumes two mutually engaged people (a street fight, a hockey brawl). The target scenario here is asymmetric: one agitated patient and one non-reciprocating caregiver, so the architecture evaluates each tracked individual's motion independently rather than jointly classifying a pair.

Using pose keypoints instead of raw video also means no raw imagery needs to be retained after extraction, which is relevant given the intended patient-monitoring application.


## Data source

Training data is drawn from RLVS (Real Life Violence Situations), a 2000-clip labeled dataset of street-level violent and non-violent footage. Two other candidate datasets were evaluated and excluded:

- **RWF-2000** was excluded because its license restricts modification and redistribution without approval from the original authors.
- **Hockey Fight Detection** was excluded after a sanity check showed severe tracking failure on that footage (tracked sequences as short as 6 to 8 frames out of a roughly 24-frame maximum clip length), most likely caused by the dataset's age, low resolution, and fast puck-game motion obscuring pose estimation.


## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install ultralytics -q
!nvidia-smi


In [ ]:
import os

if not os.path.ismount("/content/drive"):
    raise RuntimeError(
        "Google Drive did not mount successfully (the cell above should have "
        "prompted an authorization popup - if it was dismissed or it failed "
        "silently, /content/drive won't be a real mount). Every checkpoint and "
        "dataset backup in this notebook assumes Drive is mounted; proceeding "
        "without it would silently write to local, non-persistent storage and "
        "lose everything on the next disconnect. Re-run the cell above and "
        "complete the authorization, then rerun this cell to confirm."
    )
print("Google Drive confirmed mounted.")


In [ ]:
import os, json, getpass

# Prompted at runtime instead of hardcoded, so this notebook is safe to
# commit/share - get an API key from kaggle.com/settings -> API -> Create New Token.
KAGGLE_USERNAME = getpass.getpass("Kaggle username: ")
KAGGLE_KEY = getpass.getpass("Kaggle API key: ")

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
os.system('pip install --upgrade kaggle -q')


In [ ]:
import subprocess
from pathlib import Path

RLVS_DRIVE_BACKUP = "/content/drive/MyDrive/fight_detection_data/rlvs"
RLVS_LOCAL = "/content/data/rlvs"
os.makedirs("/content/data", exist_ok=True)

if Path(RLVS_DRIVE_BACKUP).exists():
    subprocess.run(["cp", "-r", RLVS_DRIVE_BACKUP, RLVS_LOCAL])
else:
    result = subprocess.run(
        ["kaggle", "datasets", "download",
         "-d", "mohamedmustafa/real-life-violence-situations-dataset",
         "-p", RLVS_LOCAL, "--unzip"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(
            "Kaggle download failed (check kaggle.json credentials in the "
            "previous cell). stderr:\n" + result.stderr
        )
    os.makedirs('/content/drive/MyDrive/fight_detection_data', exist_ok=True)
    subprocess.run(["cp", "-r", RLVS_LOCAL, RLVS_DRIVE_BACKUP])

if not Path(RLVS_LOCAL).exists():
    raise RuntimeError(f"{RLVS_LOCAL} was not created - do not proceed, "
                        "extraction will silently run on zero clips.")
print("RLVS ready.")


## Feature extraction pipeline

`extract_features.py` converts raw video into cached, normalized pose sequences.

**Tracking.** ByteTrack is used instead of the default BoT-SORT, because BoT-SORT's global motion compensation step produced repeated OpenCV assertion failures on this footage; ByteTrack does not perform that step and tracks reliably on the same data.

**Variable-length sequences.** Early versions of this pipeline used fixed 30-frame windows, which discarded any tracked person shorter than the window, including real tracked motion of 14 to 20 frames. The current version pads each track's real length up to 90 frames (about 6 seconds at 15 frames per second) and stores the true length alongside it, so the model receives the real amount of motion available rather than losing shorter but genuine sequences. Tracks shorter than 8 frames are discarded as too brief to carry meaningful motion information.

**Boundary artifact filtering.** Investigation of a TSAuditor data quality finding (detailed later in this notebook) revealed that YOLOv8-pose clamps a keypoint to the frame boundary when a body part extends off screen, rather than marking it as undetected. This produced a plausible-looking but false constant value that a simple all-zero detection-failure check did not catch. `normalize_skeleton` now also rejects any frame where a keypoint sits within one pixel of the frame edge.


In [ ]:
%%writefile extract_features.py
"""
Feature extraction pipeline: video -> tracked, normalized keypoint sequences.
"""
import numpy as np
import cv2
from ultralytics import YOLO
from pathlib import Path

TARGET_FPS = 15
MAX_SEQ_LEN = 90
MIN_SEQ_LEN = 8
NUM_KEYPOINTS = 17

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
pose_model = YOLO("yolov8n-pose.pt")
pose_model.to(DEVICE)


def resample_to_fps(cap, target_fps):
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step = src_fps / target_fps
    next_keep = 0.0
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx >= next_keep:
            yield frame
            next_keep += step
        frame_idx += 1


def normalize_skeleton(keypoints):
    """No longer rejects a frame just for touching a boundary - that was
    too broad and was pushing whole clips under MIN_SEQ_LEN. Genuine
    clamp artifacts are stripped downstream by _strip_stuck_boundary_runs."""
    if np.all(keypoints == 0):
        return None
    hip_l, hip_r = keypoints[11], keypoints[12]
    shoulder_l, shoulder_r = keypoints[5], keypoints[6]
    center = (hip_l + hip_r) / 2.0
    scale = np.linalg.norm(shoulder_l - shoulder_r) + 1e-6
    return (keypoints - center) / scale


def _strip_stuck_boundary_runs(raw_seq, norm_seq, dims_seq, min_stuck_run=6,
                                boundary_tol=1.0, val_tol=1e-3):
    """Drop only frames in a genuine clamp run: same keypoint frozen at an
    identical boundary value for >= min_stuck_run consecutive frames."""
    n = len(raw_seq)
    if n == 0:
        return norm_seq
    stuck = [False] * n
    num_kpts = raw_seq[0].shape[0]

    for k in range(num_kpts):
        run_start = None
        for i in range(1, n):
            fw, fh = dims_seq[i]
            x, y = raw_seq[i][k]
            touching = x <= boundary_tol or x >= fw - boundary_tol or y <= boundary_tol or y >= fh - boundary_tol
            same_as_prev = np.allclose(raw_seq[i][k], raw_seq[i - 1][k], atol=val_tol)
            if touching and same_as_prev:
                if run_start is None:
                    run_start = i - 1
            else:
                if run_start is not None and (i - run_start) >= min_stuck_run:
                    for j in range(run_start, i):
                        stuck[j] = True
                run_start = None
        if run_start is not None and (n - run_start) >= min_stuck_run:
            for j in range(run_start, n):
                stuck[j] = True

    return [norm_seq[i] for i in range(n) if not stuck[i]]


def extract_clip_features(video_path, track_history, max_tracks=2, max_frames=150):
    cap = cv2.VideoCapture(video_path)
    raw_sequences, norm_sequences, dims_sequences = {}, {}, {}
    for n_processed, frame in enumerate(resample_to_fps(cap, TARGET_FPS)):
        if n_processed >= max_frames:
            break
        frame_height, frame_width = frame.shape[:2]
        results = pose_model.track(frame, persist=True, verbose=False,
                                    device=DEVICE, tracker="bytetrack.yaml")[0]
        if results.boxes.id is None:
            continue
        ids = results.boxes.id.cpu().numpy().astype(int)
        kpts = results.keypoints.xy.cpu().numpy()
        for tid, kp in zip(ids, kpts):
            norm = normalize_skeleton(kp)
            if norm is None:
                continue
            raw_sequences.setdefault(tid, []).append(kp)
            norm_sequences.setdefault(tid, []).append(norm)
            dims_sequences.setdefault(tid, []).append((frame_width, frame_height))
    cap.release()

    cleaned = {
        tid: _strip_stuck_boundary_runs(raw_sequences[tid], norm_sequences[tid], dims_sequences[tid])
        for tid in norm_sequences
    }
    top_ids = sorted(cleaned, key=lambda k: len(cleaned[k]), reverse=True)[:max_tracks]
    return {tid: cleaned[tid] for tid in top_ids}


def pad_or_truncate(seq, max_len=MAX_SEQ_LEN):
    arr = np.stack(seq)
    real_len = min(len(arr), max_len)
    if len(arr) >= max_len:
        return arr[:max_len], real_len
    padded = np.zeros((max_len, NUM_KEYPOINTS, 2), dtype=arr.dtype)
    padded[:len(arr)] = arr
    return padded, real_len


def process_clip(video_path, label, out_dir):
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    clip_name = Path(video_path).stem
    try:
        seqs = extract_clip_features(video_path, {})
    except Exception as e:
        print(f"[FAIL] {clip_name}: {e}")
        return False

    wrote_any = False
    for tid, seq in seqs.items():
        if len(seq) < MIN_SEQ_LEN:
            continue
        padded, real_len = pad_or_truncate(seq)
        np.savez(out_path / f"{clip_name}_id{tid}.npz",
                 keypoints=padded, length=real_len, label=label, source_clip=clip_name)
        wrote_any = True

    if not wrote_any:
        print(f"[EMPTY] {clip_name}: no track reached MIN_SEQ_LEN={MIN_SEQ_LEN} frames")
    return wrote_any


def _restore_features_from_drive(out_dir, drive_features_backup):
    import shutil, tempfile
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    if any(out_path.glob("*.npz")):
        return
    backup = Path(drive_features_backup)
    if not backup.exists():
        print(f"No Drive feature backup found at {drive_features_backup} - starting fresh.")
        return
    print(f"Restoring extracted features from Drive backup ({drive_features_backup})...")
    with tempfile.TemporaryDirectory() as tmp:
        shutil.unpack_archive(str(backup), tmp, "zip")
        found = list(Path(tmp).rglob("*.npz"))
        for f in found:
            shutil.copy2(f, out_path / f.name)
        print(f"Restored {len(found)} feature files.")


def _assert_drive_mounted(path_str):
    import os
    if str(path_str).startswith("/content/drive") and not os.path.ismount("/content/drive"):
        raise RuntimeError(f"Google Drive is not mounted at /content/drive (tried to use {path_str}).")


def _backup_features_to_drive(out_dir, drive_features_backup):
    import shutil
    _assert_drive_mounted(drive_features_backup)
    base_name = drive_features_backup[:-4] if drive_features_backup.endswith(".zip") else drive_features_backup
    archive_path = shutil.make_archive(base_name + ".tmp", "zip", root_dir=out_dir)
    shutil.move(archive_path, drive_features_backup)


def batch_run(dataset_spec, out_dir, log_path, drive_sync_path=None, sync_every=50,
              drive_features_backup=None):
    import shutil

    if drive_features_backup:
        _restore_features_from_drive(out_dir, drive_features_backup)

    log_file = Path(log_path)
    done_status = {}
    if log_file.exists():
        for line in log_file.read_text().splitlines():
            if "\t" in line:
                key, status = line.split("\t", 1)
                done_status[key] = status

    out_path = Path(out_dir)

    def already_done(video_path_str, status):
        if status == "empty":
            return True
        clip_name = Path(video_path_str).stem
        return any(out_path.glob(f"{clip_name}_id*.npz"))

    done = {k for k, s in done_status.items() if already_done(k, s)}

    video_exts = {".mp4", ".avi", ".mov", ".mkv"}
    all_clips = []
    for folder, label in dataset_spec:
        for p in Path(folder).rglob("*"):
            if p.suffix.lower() in video_exts:
                all_clips.append((p, label))
    print(f"Found {len(all_clips)} clips total, {len(done)} already processed and verified present.")

    with open(log_file, "a") as log:
        for i, (video_path, label) in enumerate(all_clips):
            key = str(video_path)
            if key in done:
                continue
            success = process_clip(str(video_path), label, out_dir)
            log.write(f"{key}\t{'ok' if success else 'empty'}\n")

            if i > 0 and i % sync_every == 0:
                if drive_sync_path:
                    shutil.copy(log_file, drive_sync_path)
                if drive_features_backup:
                    _backup_features_to_drive(out_dir, drive_features_backup)
                    print(f"  (backed up features + log to Drive at clip {i}/{len(all_clips)})")

            if i % 50 == 0:
                status = "ok" if success else "empty/failed"
                print(f"[{i}/{len(all_clips)}] processed: {video_path.name} ({status})")

    if drive_sync_path:
        shutil.copy(log_file, drive_sync_path)
    if drive_features_backup:
        _backup_features_to_drive(out_dir, drive_features_backup)
        print("Final feature backup to Drive complete.")

## Pipeline validation

Before committing to extraction across the full dataset, a small sample of clips is processed to confirm that tracking succeeds, that keypoints form a coherent human pose, and that short tracked sequences (as low as 8 frames) are correctly retained rather than silently dropped.


In [ ]:
from extract_features import extract_clip_features, pad_or_truncate, MIN_SEQ_LEN
from pathlib import Path

def sample_from(folder, label, n, exts=(".mp4", ".avi")):
    files = [f for f in Path(folder).iterdir() if f.suffix.lower() in exts]
    return [(str(f), label) for f in files[:n]]

SAMPLE_CLIPS = (
    sample_from("/content/data/rlvs/Real Life Violence Dataset/Violence", 1, 5)
    + sample_from("/content/data/rlvs/Real Life Violence Dataset/NonViolence", 0, 5)
)

total_kept = 0
for video_path, label in SAMPLE_CLIPS:
    seqs = extract_clip_features(video_path, {})
    print(f"\n{video_path}")
    for tid, seq in seqs.items():
        kept = len(seq) >= MIN_SEQ_LEN
        total_kept += int(kept)
        print(f"  Track {tid}: {len(seq)} frames -> {'kept' if kept else 'dropped (too short)'}")

print(f"\nTotal tracks kept across {len(SAMPLE_CLIPS)} sample clips: {total_kept}")


## Full extraction

Extraction runs across all Violence and NonViolence clips in RLVS. Output is cached as one file per tracked person, containing the padded keypoint sequence, its real length, the clip's label, and the source clip name (used later for a leakage-safe train and test split).


Skip this cell to avoid re-extraction.

In [ ]:
# --- One-time reset: force clean re-extraction with the fixed normalize_skeleton ---
# Run this once if the existing features_v2 (local cache or Drive backup zip)
# predates the boundary_tol clamp-detection fix in normalize_skeleton - the
# restore/skip-if-cached logic in the next two cells would otherwise silently
# reuse that old (possibly boundary-clamped) data instead of regenerating it.
# Wipes local cached features, the Drive backup zip, and both extraction logs
# so the next two cells are forced to run full extraction through the current
# (fixed) extract_features.py. Do NOT run this on every restart - only once,
# now, to clear out data that predates the fix.
import shutil
from pathlib import Path

FEATURES_LOCAL = Path("/content/features_v2")
FEATURES_BACKUP_ZIP = Path("/content/drive/MyDrive/features_v2_backup.zip")
LOG_LOCAL = Path("/content/extraction_log_v2.txt")
LOG_DRIVE = Path("/content/drive/MyDrive/extraction_log_v2.txt")

removed = []
if FEATURES_LOCAL.exists():
    shutil.rmtree(FEATURES_LOCAL)
    removed.append(str(FEATURES_LOCAL))
if FEATURES_BACKUP_ZIP.exists():
    FEATURES_BACKUP_ZIP.unlink()
    removed.append(str(FEATURES_BACKUP_ZIP))
if LOG_LOCAL.exists():
    LOG_LOCAL.unlink()
    removed.append(str(LOG_LOCAL))
if LOG_DRIVE.exists():
    LOG_DRIVE.unlink()
    removed.append(str(LOG_DRIVE))

if removed:
    print("Cleared for clean re-extraction:")
    for r in removed:
        print(f"  - {r}")
else:
    print("Nothing to clear - no existing features_v2, backup zip, or logs found.")


In [ ]:
import zipfile, shutil, tempfile
from pathlib import Path

FEATURES_LOCAL = Path("/content/features_v2")
FEATURES_BACKUP_ZIP = Path("/content/drive/MyDrive/features_v2_backup.zip")

if FEATURES_LOCAL.exists() and any(FEATURES_LOCAL.glob("*.npz")):
    print(f"{FEATURES_LOCAL} already has cached features, nothing to restore.")
elif FEATURES_BACKUP_ZIP.exists():
    print("Restoring cached features from Drive backup...")
    with tempfile.TemporaryDirectory() as tmp:
        with zipfile.ZipFile(FEATURES_BACKUP_ZIP) as z:
            z.extractall(tmp)
        # Don't assume a fixed internal layout - `zip -r` on an absolute
        # path stores members with the leading slash stripped (e.g.
        # content/features_v2/...), which is easy to extract to the wrong
        # place. Find every .npz file wherever it landed and copy it
        # directly into FEATURES_LOCAL instead of guessing the path.
        found = list(Path(tmp).rglob("*.npz"))
        FEATURES_LOCAL.mkdir(parents=True, exist_ok=True)
        for f in found:
            shutil.copy2(f, FEATURES_LOCAL / f.name)
        print(f"Restored {len(found)} feature files.")
    if not any(FEATURES_LOCAL.glob("*.npz")):
        print(f"warning: {FEATURES_BACKUP_ZIP} existed but contained no .npz files. "
              "Run `!unzip -l " + str(FEATURES_BACKUP_ZIP) + "` to inspect it directly "
              "before assuming full extraction needs to rerun.")
else:
    print("No cached features found locally or on Drive - full extraction will run next.")


In [ ]:
from extract_features import batch_run

RLVS_LOCAL = "/content/data/rlvs"
FEATURES_DIR = "/content/features_v2"

existing_npz = list(Path(FEATURES_DIR).glob("*.npz")) if Path(FEATURES_DIR).exists() else []

if len(existing_npz) > 500:
    # 2002 train + 670 val + 673 test = 3345 in the original run; 500 is a
    # conservative floor that only matters if the restore step above found
    # nothing and extraction genuinely needs to run.
    print(f"Found {len(existing_npz)} cached feature files, skipping extraction.")
else:
    rlvs_spec = [
        (f"{RLVS_LOCAL}/Real Life Violence Dataset/Violence", 1),
        (f"{RLVS_LOCAL}/Real Life Violence Dataset/NonViolence", 0),
    ]
    batch_run(
        rlvs_spec,
        out_dir=FEATURES_DIR,
        log_path="/content/extraction_log_v2.txt",
        drive_sync_path="/content/drive/MyDrive/extraction_log_v2.txt",
        drive_features_backup="/content/drive/MyDrive/features_v2_backup.zip",
        sync_every=50,
    )


## Class balance

In [ ]:
import numpy as np
from collections import Counter

labels = [int(np.load(f)["label"]) for f in Path("/content/features_v2").glob("*.npz")]
counts = Counter(labels)
total = len(labels)
print(f"Total sequences: {total}")
print(f"Non-violence: {counts[0]} ({counts[0]/total*100:.1f}%)")
print(f"Violence: {counts[1]} ({counts[1]/total*100:.1f}%)")

os.system("zip -r -q /content/drive/MyDrive/features_v2_backup.zip /content/features_v2")


## Model architecture and training methodology

`train_stage1.py` implements the classifier and its evaluation.

**Model.** A single-layer LSTM (hidden size 64) takes the flattened 34-value keypoint vector (17 keypoints, x and y) at each timestep and produces a two-class prediction (aggressive or calm) from its final hidden state. Sequences are packed with `pack_padded_sequence`, so the LSTM mathematically ignores padded timesteps rather than treating them as real, still input.

**Data split.** Sequences are split 60 percent train, 20 percent validation, 20 percent test, grouped by source clip so that no clip's tracked people appear in more than one split. This prevents the model from learning clip-specific artifacts and reporting inflated validation performance.

**Training.** The validation set selects the best checkpoint and drives early stopping (training halts after 8 epochs without improvement). The test set is held out entirely until training is complete, and is evaluated exactly once.

**Evaluation.** The held-out test set is scored with accuracy, precision, recall, F1, ROC-AUC, and a full confusion matrix, all saved alongside plots for reporting.


In [ ]:
%%writefile train_stage1.py
"""
Stage 1: 2-class (aggressive / calm) per-person motion classifier.

What the features are:
Each training example is a sequence of normalized 2D human pose keypoints
for one tracked person over time (not raw pixels, not bounding boxes
alone). Specifically: 17 COCO-format joints (nose, eyes, ears, shoulders,
elbows, wrists, hips, knees, ankles), each as an (x, y) coordinate,
extracted per frame by YOLOv8-pose and linked across frames by ByteTrack
so the sequence follows the same person over time.

Why this matters for violence/non-violence classification:
A single frame can't distinguish a punch mid-swing from a wave, both
are "arm raised, elbow bent" in one static image. What separates
aggressive motion from calm motion is the trajectory of joints over
time: velocity, acceleration, and the specific limb configurations that
occur in sequence (e.g. rapid arm extension toward another person's
position, versus slow symmetric arm movement in walking). The keypoints
are normalized (centered on hip midpoint, scaled by shoulder width) so
the model learns motion shape independent of the person's distance from
camera or frame position - standard practice in pose-based action
recognition (see: ST-GCN, ntu-rgb+d skeleton action recognition
literature), not raw pixel/appearance-based classification.

The model is validated on the same feature space it's trained on:
there is no separate "pose vs features" split in this pipeline, the
pose keypoints are the features. There is currently no second feature
stream (e.g. optical flow, raw bounding box motion, or appearance/CNN
features) combined with the pose stream; that's a scoping choice,
discussed as an open extension below, not an oversight.

Uses pack_padded_sequence so the LSTM ignores padded frames instead of
treating them as real (zero-motion) input.

Resumable: checkpoints saved to Drive after every epoch. Includes a
proper held-out test set (not just train/val) with full evaluation:
confusion matrix, F1, ROC-AUC, precision-recall curve.
"""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (confusion_matrix, f1_score, roc_auc_score,
                              roc_curve, precision_recall_curve, classification_report)
import matplotlib.pyplot as plt
import json

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Fails loudly if Drive isn't mounted yet, rather than silently creating a
# local /content/drive/... folder that looks identical to a real Drive path
# but isn't persistent - checkpoints saved there would vanish on the next
# disconnect with no error at save time to flag it.
import os
if not os.path.ismount("/content/drive"):
    raise RuntimeError(
        "Google Drive is not mounted at /content/drive. Run the Drive-mount "
        "cell first - importing this module before that would silently "
        "create local (non-persistent) checkpoint/results folders instead "
        "of real Drive ones."
    )

CHECKPOINT_DIR = Path("/content/drive/MyDrive/fight_detection_checkpoints")
CHECKPOINT_PATH = CHECKPOINT_DIR / "lstm_stage1.pt"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = Path("/content/drive/MyDrive/fight_detection_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

NUM_KEYPOINTS = 17
INPUT_DIM = NUM_KEYPOINTS * 2
HIDDEN_DIM = 64
NUM_LAYERS = 1
BATCH_SIZE = 32
TOTAL_EPOCHS = 50
LR = 1e-3
PATIENCE = 8


class KeypointSeqDataset(Dataset):
    def __init__(self, npz_paths):
        self.paths = npz_paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        data = np.load(self.paths[idx])
        kp = data["keypoints"].reshape(data["keypoints"].shape[0], -1)
        length = int(data["length"])
        label = int(data["label"])
        return (torch.tensor(kp, dtype=torch.float32),
                torch.tensor(length, dtype=torch.long),
                torch.tensor(label, dtype=torch.long))


class MotionLSTM(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, num_classes=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        last_hidden = h_n[-1]
        return self.classifier(last_hidden)


def three_way_split(npz_dir, test_fraction=0.2, val_fraction=0.2, seed=42):
    """
    Leakage-safe 60/20/20 train/val/test split, grouped by source_clip so
    a clip's tracks never appear in more than one split.
    """
    all_files = list(Path(npz_dir).glob("*.npz"))
    groups = [str(np.load(f, allow_pickle=True)["source_clip"]) for f in all_files]

    # first peel off the test set
    splitter1 = GroupShuffleSplit(n_splits=1, test_size=test_fraction, random_state=seed)
    idx = np.arange(len(all_files))
    trainval_idx, test_idx = next(splitter1.split(idx, groups=groups))

    # split remaining into train/val, val_fraction is relative to the
    # ORIGINAL total, so recompute relative to the trainval remainder
    val_relative = val_fraction / (1 - test_fraction)
    trainval_groups = [groups[i] for i in trainval_idx]
    splitter2 = GroupShuffleSplit(n_splits=1, test_size=val_relative, random_state=seed)
    train_rel_idx, val_rel_idx = next(splitter2.split(trainval_idx, groups=trainval_groups))

    train_idx = trainval_idx[train_rel_idx]
    val_idx = trainval_idx[val_rel_idx]

    train_files = [all_files[i] for i in train_idx]
    val_files = [all_files[i] for i in val_idx]
    test_files = [all_files[i] for i in test_idx]
    return train_files, val_files, test_files


def save_checkpoint(model, optimizer, epoch, best_val_acc):
    torch.save({"epoch": epoch, "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_val_acc": best_val_acc}, CHECKPOINT_PATH)


def load_checkpoint(model, optimizer):
    if not CHECKPOINT_PATH.exists():
        return 0, 0.0
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    print(f"Resuming from epoch {ckpt['epoch']} (best val acc so far: {ckpt['best_val_acc']:.4f})")
    return ckpt["epoch"], ckpt["best_val_acc"]


def evaluate(model, loader):
    """Quick per-epoch metrics used during training (accuracy/precision/recall)."""
    model.eval()
    correct, total = 0, 0
    tp, fp, fn = 0, 0, 0
    with torch.no_grad():
        for x, lengths, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            preds = model(x, lengths).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
            tp += ((preds == 1) & (y == 1)).sum().item()
            fp += ((preds == 1) & (y == 0)).sum().item()
            fn += ((preds == 0) & (y == 1)).sum().item()
    acc = correct / max(total, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    return acc, precision, recall


def evaluate_test_full(model, test_loader, save_prefix="stage1"):
    """
    Full evaluation on the held-out test set (never seen during training
    or checkpoint selection): confusion matrix, F1, ROC-AUC, PR curve.
    Saves plots + a JSON metrics summary to Drive.
    """
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for x, lengths, y in test_loader:
            x = x.to(DEVICE)
            logits = model(x, lengths)
            probs = F.softmax(logits, dim=1)[:, 1]  # P(violence)
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    cm = confusion_matrix(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)
    report = classification_report(all_labels, all_preds,
                                    target_names=["non-violence", "violence"],
                                    output_dict=True)

    print("\n=== Test set results (held-out, never used for training or checkpointing) ===")
    print(f"F1 score: {f1:.4f}")
    print(f"ROC-AUC: {auc:.4f}")
    print("\nConfusion matrix:")
    print(f"                 pred_non-violence  pred_violence")
    print(f"true_non-violence      {cm[0][0]:>6}          {cm[0][1]:>6}")
    print(f"true_violence          {cm[1][0]:>6}          {cm[1][1]:>6}")
    print("\nFull classification report:")
    print(classification_report(all_labels, all_preds, target_names=["non-violence", "violence"]))

    # confusion matrix plot
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["non-violence", "violence"])
    ax.set_yticklabels(["non-violence", "violence"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix, Test Set (F1={f1:.3f})")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i][j], ha="center", va="center",
                     color="white" if cm[i][j] > cm.max()/2 else "black")
    plt.colorbar(im)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"{save_prefix}_confusion_matrix.png", dpi=150)
    plt.show()

    # ROC curve
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, label=f"ROC (AUC={auc:.3f})")
    plt.plot([0, 1], [0, 1], "k--", alpha=0.3)
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC Curve, Test Set")
    plt.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"{save_prefix}_roc_curve.png", dpi=150)
    plt.show()

    # Precision-Recall curve
    prec, rec, _ = precision_recall_curve(all_labels, all_probs)
    plt.figure(figsize=(5, 4))
    plt.plot(rec, prec)
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title("Precision-Recall Curve, Test Set")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"{save_prefix}_pr_curve.png", dpi=150)
    plt.show()

    # save metrics summary for cross-experiment comparison (per supervisor's
    # request for a spreadsheet comparing multiple runs/splits/pipelines)
    summary = {
        "f1": f1, "roc_auc": auc,
        "confusion_matrix": cm.tolist(),
        "accuracy": report["accuracy"],
        "precision_violence": report["violence"]["precision"],
        "recall_violence": report["violence"]["recall"],
        "precision_nonviolence": report["non-violence"]["precision"],
        "recall_nonviolence": report["non-violence"]["recall"],
        "n_test_samples": len(all_labels),
    }
    with open(RESULTS_DIR / f"{save_prefix}_test_metrics.json", "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\nMetrics + plots saved to {RESULTS_DIR}")
    return summary


def train(npz_dir, save_prefix="stage1"):
    train_files, val_files, test_files = three_way_split(npz_dir)
    print(f"Train: {len(train_files)}  Val: {len(val_files)}  Test: {len(test_files)}  "
          f"(60/20/20 split, grouped by source_clip, leakage-safe)")

    train_loader = DataLoader(KeypointSeqDataset(train_files), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(KeypointSeqDataset(val_files), batch_size=BATCH_SIZE)
    test_loader = DataLoader(KeypointSeqDataset(test_files), batch_size=BATCH_SIZE)

    model = MotionLSTM().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    start_epoch, best_val_acc = load_checkpoint(model, optimizer)
    epochs_without_improvement = 0

    for epoch in range(start_epoch, TOTAL_EPOCHS):
        model.train()
        total_loss = 0.0
        for x, lengths, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x, lengths)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)

        avg_loss = total_loss / max(len(train_files), 1)
        val_acc, val_prec, val_rec = evaluate(model, val_loader)
        print(f"Epoch {epoch+1}/{TOTAL_EPOCHS} | loss {avg_loss:.4f} | "
              f"val_acc {val_acc:.4f} | val_precision {val_prec:.4f} | val_recall {val_rec:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_without_improvement = 0
            best_path = CHECKPOINT_DIR / f"lstm_{save_prefix}_best.pt"
            torch.save({"epoch": epoch + 1, "model_state": model.state_dict(),
                        "val_acc": val_acc}, best_path)
            print(f"  -> new best (val_acc={val_acc:.4f}), saved to {best_path.name}")
        else:
            epochs_without_improvement += 1

        save_checkpoint(model, optimizer, epoch + 1, best_val_acc)

        if epochs_without_improvement >= PATIENCE:
            print(f"\nNo improvement for {PATIENCE} epochs, stopping early at epoch {epoch+1}.")
            break

    print(f"\nTraining complete. Best val acc: {best_val_acc:.4f}")

    # load the best checkpoint (not necessarily the last epoch) before
    # final test evaluation - that's the model that actually generalized
    best_path = CHECKPOINT_DIR / f"lstm_{save_prefix}_best.pt"
    best_ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(best_ckpt["model_state"])

    evaluate_test_full(model, test_loader, save_prefix=save_prefix)

    return model


## Training

In [ ]:
from pathlib import Path
import torch
from train_stage1 import train, MotionLSTM, DEVICE

BEST_CKPT = Path("/content/drive/MyDrive/fight_detection_checkpoints/lstm_stage1_best.pt")

if BEST_CKPT.exists():
    print(f"Found existing checkpoint at {BEST_CKPT}, skipping training and loading it instead.")
    model = MotionLSTM().to(DEVICE)
    ckpt = torch.load(BEST_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    print(f"Loaded (val_acc={ckpt.get('val_acc', 'n/a')}). "
          "Delete this checkpoint file first to retrain from scratch.")
else:
    model = train(npz_dir="/content/features_v2")


## Results and analysis

Final held-out test set results (2002 train / 670 validation / 673 test sequences, trained on the boundary-artifact-corrected dataset):

| Metric | Value |
|---|---|
| F1 score | 0.7874 |
| ROC-AUC | 0.8435 |
| Test accuracy | 0.76 |
| Validation accuracy (checkpoint selection) | 0.8164 |

**Confusion matrix (test set, n=673):**

| | Predicted non-violence | Predicted violence |
|---|---|---|
| True non-violence | 211 | 83 |
| True violence | 79 | 300 |

**Per-class precision and recall:**

| Class | Precision | Recall | F1 |
|---|---|---|---|
| Non-violence | 0.73 | 0.72 | 0.72 |
| Violence | 0.78 | 0.79 | 0.79 |

**Analysis.** Test accuracy (0.76) is notably lower than the validation accuracy used for checkpoint selection (0.8164). This gap is larger than typical and is attributable to three compounding factors rather than a single cause:

1. **Dataset size.** With 2002 training sequences split across roughly 2000 source clips, each split (particularly the 673-sequence test set) is small enough that individual clips carry meaningful statistical weight. A handful of harder or unusually-framed clips landing in the test split can shift accuracy by a few points in a way that would average out in a larger dataset.
2. **Grouped splitting variance.** Because the split is grouped by source clip (a deliberate choice to prevent leakage, not something to relax), an entire clip's full signal, easy or hard, sits in exactly one split. This is the correct methodology, but it trades away the smoothing effect that random per-sequence splitting would produce, at the cost of a noisier val-test gap.
3. **Early stopping selects on validation, not test.** The checkpoint kept is whichever epoch scored best on validation. Some of that validation score reflects the specific epoch happening to fit the validation split's particular composition well, which does not fully transfer to a separate held-out set. This is expected behavior, not a sign of a broken training process, but it does mean validation accuracy should not be quoted as the model's real-world performance estimate. Test-set metrics should be.

Compared to an earlier baseline trained before the leakage-safe split and full test evaluation existed, precision and recall are now much more balanced across both classes (previously recall notably outpaced precision, meaning the model over-triggered on violence). This matters directly for the target deployment: false positives in a caregiver-alerting system cause alert fatigue, so a more balanced precision and recall profile is a meaningful improvement independent of the headline accuracy figure.


## Data quality audit with TSAuditor

TSAuditor is a time-series data quality auditing library, applied here to a domain outside its original scope: per-person pose keypoint sequences rather than financial or sensor data.

**Approach.** Each tracked person's sequence was converted into its own DataFrame (indexed by a synthetic timestamp derived from frame number and frame rate) and audited independently with `tsa.scan()`, dispatched in parallel across all 3600-plus tracks using joblib. The `target` parameter was left unset, since leakage detection requires a per-frame prediction target, which does not apply here (each track has a single label for the whole sequence, not a value that varies frame to frame).

**Finding.** The audit flagged 1099 ANO001 (stuck value) findings, almost entirely on vertical coordinate columns, with some joints reporting an unchanging value for over 50 consecutive frames. Direct inspection of raw, pre-normalization keypoints for a flagged track confirmed the cause: a body part extending below the visible frame was having its vertical coordinate clamped to the frame boundary by YOLOv8-pose, rather than being marked as undetected. Because the resulting value was not `(0, 0)`, it passed through the pipeline's existing all-zero detection-failure filter undetected, and produced a normal-looking skeleton in visual inspection despite being incorrect.

**Fix and verification.** `normalize_skeleton` was updated to reject any keypoint within one pixel of the frame boundary. The dataset was re-extracted and re-audited: the same check now reports zero ANO001 findings across the corrected dataset.


In [ ]:
!pip install tsauditor -q

In [ ]:
%%writefile tsauditor_check.py
"""
Convert cached pose sequences (.npz) into per-track DataFrames, then audit
them in parallel using the joblib pattern from TSAuditor's own README:
this specifically requires separate DataFrames per entity (not a single
long-format frame with group_col), since that's the code path the
library's joblib parallelization actually targets.

target is None: leakage detection needs a per-frame target, which doesn't
apply here (each track has one scalar label for the whole sequence).
"""
import numpy as np
import pandas as pd
from pathlib import Path
from joblib import Parallel, delayed
import tsauditor as tsa

FEATURES_DIR = "/content/features_v2"
FPS = 15


def npz_to_track_frames(features_dir: str) -> dict:
    """Returns {track_id: per-track DataFrame}, separate frames, not one
    combined long-format frame, so the joblib pattern applies cleanly."""
    frames = {}
    for f in Path(features_dir).glob("*.npz"):
        data = np.load(f, allow_pickle=True)
        kp = data["keypoints"]
        length = int(data["length"])
        track_id = f.stem
        real_frames = kp[:length]

        rows = []
        for t, frame_kp in enumerate(real_frames):
            row = {"frame_idx": t}
            for j in range(17):
                row[f"kp{j}_x"] = frame_kp[j, 0]
                row[f"kp{j}_y"] = frame_kp[j, 1]
            rows.append(row)

        df = pd.DataFrame(rows)
        df["timestamp"] = pd.Timestamp("2000-01-01") + pd.to_timedelta(df["frame_idx"] / FPS, unit="s")
        df = df.set_index("timestamp").drop(columns=["frame_idx"])
        frames[track_id] = df

    return frames


def audit_track(track_id, df):
    report = tsa.scan(df, target=None, domain="sensor",
                       run_leakage=False, run_stationarity=False)
    return track_id, report


if __name__ == "__main__":
    print("Building per-track DataFrames...")
    frames = npz_to_track_frames(FEATURES_DIR)
    print(f"{len(frames)} tracks ready for parallel audit.")

    import time
    t0 = time.time()
    results = dict(Parallel(n_jobs=-1)(
        delayed(audit_track)(track_id, df) for track_id, df in frames.items()
    ))
    print(f"Parallel scan of {len(frames)} tracks took {time.time()-t0:.1f}s")

    # collect ANO001 (stuck value) findings across all tracks
    stuck = []
    for track_id, report in results.items():
        for issue in (report.critical + report.warnings + report.info):
            if "ANO001" in str(getattr(issue, "code", "")):
                stuck.append((track_id, issue))

    print(f"\n{len(stuck)} ANO001 (stuck value) findings:")
    for track_id, issue in stuck[:20]:
        print(f"  {track_id}: {issue}")


In [ ]:
%run tsauditor_check.py

## Summary

A per-person, pose-based LSTM classifier was built and evaluated on RLVS, using a leakage-safe grouped train, validation, and test split and full held-out test evaluation. A data quality audit with TSAuditor identified a real extraction bug (frame-boundary keypoint clamping) that a visual sanity check had not caught. The bug was fixed, the dataset was re-extracted and re-verified as clean, and the model was retrained on the corrected data, producing an F1 score of 0.7874 and ROC-AUC of 0.8435 on a held-out test set with a more balanced precision and recall profile than the earlier baseline.


## Real-world video evaluation

Everything above evaluates on a held-out split of RLVS: F1=0.7874, ROC-AUC=0.8435,
same distribution as training (street-level violence). That number confirms the
model generalizes *within* RLVS. It says nothing about generalization to footage
that looks different from RLVS, which matters here because the actual target
application is patient/caregiver contact violence in care settings (patient hits
caregiver, or the reverse), not street fighting.

Two separate checks below, and they answer different questions:

1. **Cross-domain camera check (UBI-Fights).** Real fixed-camera CCTV footage
   instead of RLVS's handheld street video. Labels come free from the dataset's
   own filename convention, so no manual work. This tests whether the model's
   predictions hold up on a different camera domain. It does NOT test the
   patient/caregiver population gap - every video in UBI-Fights (like RLVS,
   UCF-Crime, XD-Violence) is still able-bodied adult vs. able-bodied adult.

2. **Target-domain check (hand-labeled clips).** A small (15-20 clip) set matching
   the actual deployment scenario as closely as possible - weaker/restrained
   motion, care-setting camera angle. This is the number that actually matters
   for deployment. No public dataset exists for this (real patient/caregiver
   violence footage is, for good reason, not something that gets published), so
   this set has to be hand-labeled.

`evaluate_on_videos.py` reuses `extract_clip_features` from `extract_features.py`
unchanged, so inference-time preprocessing exactly matches training (same
normalization, same boundary-clamp filtering, same tracker). Building a separate
extraction path for inference is a common silent bug - any mismatch there
degrades predictions in a way that looks like a model problem but isn't.

In [ ]:
%%writefile evaluate_on_videos.py
"""
Run the trained MotionLSTM on new video files and report predictions.

What this does, versus the existing RLVS test evaluation
----------------------------------------------------------
The notebook's existing "Results and analysis" section (F1=0.7874,
ROC-AUC=0.8435) is a held-out test split of RLVS: same distribution
(street-level violence clips) as training, just clips the model never
trained/checkpointed on. That number shows how well the model
generalizes within the RLVS distribution. It says nothing about how the
model behaves on footage that looks different from RLVS - e.g. actual
CCTV footage, or footage resembling the eventual target application
(agitation in dementia/Parkinson's patients: usually a single person,
slower and non-contact motion, different camera height/angle, different
lighting). That's a domain-generalization question, and RLVS test
accuracy can't answer it.

So "check it performs on videos" has two different meanings that
shouldn't be conflated:

1. Confirm the RLVS test number is real (rerun evaluate_test_full on
   the test split) - sanity check, not new information.
2. Run inference on other videos outside RLVS, ideally footage close to
   the real deployment domain, and manually judge the predictions
   (and/or compute metrics against a small hand-labeled out-of-domain
   set). This script does (2). It reuses extract_clip_features from
   extract_features.py so the preprocessing exactly matches training
   (same normalization, same boundary-clamp filtering, same tracker).
   Using a different extraction path for inference than training is a
   common silent bug - any mismatch (e.g. skipping the boundary-clamp
   filter, or normalizing differently) degrades predictions in a way
   that looks like a model problem but is actually a preprocessing
   mismatch.

Usage
-----
    python evaluate_on_videos.py --videos /path/to/video_dir --labels labels.csv
    python evaluate_on_videos.py --videos /path/to/video_dir   # no labels: predictions only

labels.csv (optional) format: filename,label   where label is 0 or 1.
Only provide this if real ground truth exists for these videos -
guessed labels would just make the numbers look better without meaning
anything.
"""
import argparse
import csv
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

from extract_features import extract_clip_features, pad_or_truncate, MIN_SEQ_LEN, MAX_SEQ_LEN
from train_stage1 import MotionLSTM, DEVICE

CHECKPOINT_PATH = Path("/content/drive/MyDrive/fight_detection_checkpoints/lstm_stage1_best.pt")
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}


def load_model(checkpoint_path=CHECKPOINT_PATH):
    model = MotionLSTM().to(DEVICE)
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    print(f"Loaded checkpoint from {checkpoint_path} (val_acc={ckpt.get('val_acc', 'n/a')})")
    return model


def predict_video(model, video_path):
    """Returns a list of (track_id, prob_violence, real_length) for
    every usable tracked person in the video. A video can have more
    than one prediction if more than one person is tracked - how to
    combine them is a deliberate choice (see aggregate_video_prediction),
    not just taking the first one."""
    seqs = extract_clip_features(video_path, {})
    results = []
    for tid, seq in seqs.items():
        if len(seq) < MIN_SEQ_LEN:
            continue
        padded, real_len = pad_or_truncate(seq, max_len=MAX_SEQ_LEN)
        x = torch.tensor(padded.reshape(padded.shape[0], -1), dtype=torch.float32).unsqueeze(0).to(DEVICE)
        lengths = torch.tensor([real_len], dtype=torch.long)
        with torch.no_grad():
            logits = model(x, lengths)
            prob = F.softmax(logits, dim=1)[0, 1].item()
        results.append((tid, prob, real_len))
    return results


def aggregate_video_prediction(track_results, threshold=0.5):
    """A video is flagged violent if any tracked person's sequence is
    classified violent. This is a deliberate choice, not a default:
    fights involve at least one aggressor, so max-aggregation matches
    the semantics better than averaging across tracked people (which
    would dilute a clear aggressor's signal with a bystander's calm
    signal). If no usable track was extracted, return None - defaulting
    silently to "non-violent" would hide extraction failures."""
    if not track_results:
        return None, None
    max_prob = max(p for _, p, _ in track_results)
    return int(max_prob >= threshold), max_prob


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--videos", required=True, help="Directory of video files to run inference on")
    parser.add_argument("--labels", default=None, help="Optional CSV: filename,label (0/1 ground truth)")
    parser.add_argument("--checkpoint", default=str(CHECKPOINT_PATH))
    parser.add_argument("--threshold", type=float, default=0.5)
    parser.add_argument("--out_csv", default="video_predictions.csv")
    args = parser.parse_args()

    ground_truth = {}
    if args.labels:
        with open(args.labels) as f:
            for row in csv.reader(f):
                if len(row) < 2:
                    continue
                try:
                    ground_truth[row[0]] = int(row[1])
                except ValueError:
                    continue  # skips a header row (e.g. "file,label") silently

    model = load_model(args.checkpoint)

    video_dir = Path(args.videos)
    video_files = sorted(p for p in video_dir.iterdir() if p.suffix.lower() in VIDEO_EXTS)
    print(f"Found {len(video_files)} videos in {video_dir}")

    rows = []
    for vp in video_files:
        track_results = predict_video(model, str(vp))
        pred, max_prob = aggregate_video_prediction(track_results, threshold=args.threshold)
        if pred is None:
            print(f"[no usable track] {vp.name} - extraction failed or every track < {MIN_SEQ_LEN} frames")
            rows.append({"file": vp.name, "n_tracks": 0, "pred": "", "max_prob": "",
                         "label": ground_truth.get(vp.name, "")})
            continue

        label = ground_truth.get(vp.name, "")
        correct = "" if label == "" else ("correct" if pred == label else "wrong")
        print(f"{vp.name}: pred={'violence' if pred else 'non-violence'} "
              f"(p={max_prob:.3f}, {len(track_results)} track(s)) {('label=' + str(label) + ' ' + correct) if label != '' else ''}")
        rows.append({"file": vp.name, "n_tracks": len(track_results), "pred": pred,
                     "max_prob": round(max_prob, 4), "label": label})

    with open(args.out_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["file", "n_tracks", "pred", "max_prob", "label"])
        writer.writeheader()
        writer.writerows(rows)
    print(f"\nSaved per-video predictions to {args.out_csv}")

    if ground_truth:
        labeled = [r for r in rows if r["label"] != "" and r["pred"] != ""]
        if labeled:
            correct = sum(1 for r in labeled if r["pred"] == r["label"])
            acc = correct / len(labeled)
            print(f"\nAccuracy on {len(labeled)} labeled videos: {acc:.4f}")
            print("This is a separate number from the RLVS test F1/ROC-AUC. "
                  "Both should be reported, not averaged or used as a substitute for the other.")


if __name__ == "__main__":
    main()


### Cross-domain check: getting UBI-Fights via Kaggle

Community mirror: `intissarziani/ubi-fightsall`. Uses the same `kaggle` CLI /
token setup as the RLVS download above. UBI-Fights is 1,000 videos / 80 hours --
far more than needed for an eval sanity check, so pull a balanced subsample
instead of running extraction over the whole thing.

Labels come from the dataset's own naming convention (`F_..._` = fight,
`N_..._` = normal) - confirm a few filenames match this convention on the
Kaggle mirror before trusting it, since it's an unofficial re-upload.

In [ ]:
import subprocess
import shutil
import tempfile
import os
from pathlib import Path

UBI_RAW = Path("/content/data/ubi_fights_raw")
UBI_RAW_DRIVE_BACKUP = "/content/drive/MyDrive/ubi_fights_raw_backup.zip"

def restore_ubi_raw_from_drive():
    if UBI_RAW.exists() and any(UBI_RAW.rglob("*.mp4")):
        return True
    backup = Path(UBI_RAW_DRIVE_BACKUP)
    if not backup.exists():
        return False
    print(f"Restoring UBI-Fights raw videos from Drive backup ({UBI_RAW_DRIVE_BACKUP})...")
    UBI_RAW.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory() as tmp:
        shutil.unpack_archive(str(backup), tmp, "zip")
        found = list(Path(tmp).rglob("*.mp4"))
        for f in found:
            shutil.copy2(f, UBI_RAW / f.name)
    n = len(list(UBI_RAW.rglob("*.mp4")))
    print(f"Restored {n} videos.")
    return n > 0

if restore_ubi_raw_from_drive():
    pass
else:
    print("Downloading from Kaggle...")
    subprocess.run(["kaggle", "datasets", "files", "intissarziani/ubi-fightsall"])
    result = subprocess.run(
        ["kaggle", "datasets", "download", "-d", "intissarziani/ubi-fightsall",
         "-p", str(UBI_RAW), "--unzip"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError("UBI-Fights download failed: " + result.stderr)
    n = len(list(UBI_RAW.rglob("*.mp4")))
    print(f"Downloaded {n} videos.")

    # Back up to Drive so a future disconnect restores instead of
    # re-downloading. Note: this dataset can be several GB, so on tight
    # Drive storage this step is skipped safely below rather than
    # silently writing to local storage if Drive isn't mounted.
    if os.path.ismount("/content/drive"):
        print("Backing up raw videos to Drive (this can take a while for a large dataset)...")
        base_name = UBI_RAW_DRIVE_BACKUP[:-4] if UBI_RAW_DRIVE_BACKUP.endswith(".zip") else UBI_RAW_DRIVE_BACKUP
        archive_path = shutil.make_archive(base_name + ".tmp", "zip", root_dir=str(UBI_RAW))
        shutil.move(archive_path, UBI_RAW_DRIVE_BACKUP)
        print(f"Backed up to {UBI_RAW_DRIVE_BACKUP}")
    else:
        print("Google Drive not mounted - skipping backup (would otherwise silently "
              "write to local, non-persistent storage). Run the Drive-mount cell "
              "first for this to survive a disconnect.")


In [ ]:
from pathlib import Path
import shutil, zipfile

src = Path("/content/drive/MyDrive/ubi_fights_raw_backup.tmp.zip")
dst = Path("/content/drive/MyDrive/ubi_fights_raw_backup.zip")

# verify it's a complete, readable archive before trusting it
with zipfile.ZipFile(src) as z:
    bad = z.testzip()
    n = len([n for n in z.namelist() if n.lower().endswith(".mp4")])
print("corrupt member:", bad, "| mp4 count:", n)

if bad is None and n > 0:
    shutil.move(str(src), str(dst))
    print("renamed ->", dst.name)
else:
    print("archive incomplete - let the download finish instead")

In [ ]:
%%writefile subsample_ubi_fights.py
"""
Pull a small, balanced eval subset out of a full UBI-Fights download,
instead of running the pipeline over all 1,000 videos.

Assumes the standard UBI-Fights naming convention (F_ = fight,
N_ = normal). If this Kaggle mirror uses different filenames, list a
few files first and adjust label_from_filename in
parse_ubi_fights_labels.py to match - don't assume the convention
holds without checking, since this is an unofficial re-upload.

Usage:
    python subsample_ubi_fights.py --src /path/to/full_download \
        --dst /path/to/eval_subset --n_per_class 30
"""
import argparse
import random
import shutil
from pathlib import Path

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--src", required=True, help="Directory with the full downloaded dataset")
    parser.add_argument("--dst", required=True, help="Directory to copy the sampled subset into")
    parser.add_argument("--n_per_class", type=int, default=30,
                         help="How many fight and how many normal clips to keep")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    random.seed(args.seed)
    src = Path(args.src)
    dst = Path(args.dst)
    dst.mkdir(parents=True, exist_ok=True)

    all_files = [p for p in src.rglob("*") if p.suffix.lower() in VIDEO_EXTS]
    fight_files = [p for p in all_files if p.name.startswith("F_")]
    normal_files = [p for p in all_files if p.name.startswith("N_")]
    other_files = [p for p in all_files if not (p.name.startswith("F_") or p.name.startswith("N_"))]

    print(f"Found {len(all_files)} videos total: {len(fight_files)} fight, "
          f"{len(normal_files)} normal, {len(other_files)} unrecognized naming.")
    if other_files:
        print(f"warning: {len(other_files)} files don't match F_/N_ convention, e.g.: "
              f"{[p.name for p in other_files[:5]]}")
        print("Check the actual naming convention on this mirror before trusting labels.")

    n = args.n_per_class
    if len(fight_files) < n or len(normal_files) < n:
        print(f"warning: requested {n} per class but only have "
              f"{len(fight_files)} fight / {len(normal_files)} normal available.")
    sample = random.sample(fight_files, min(n, len(fight_files))) + \
             random.sample(normal_files, min(n, len(normal_files)))

    for p in sample:
        shutil.copy2(p, dst / p.name)

    print(f"\nCopied {len(sample)} videos to {dst}")
    print("Next: python parse_ubi_fights_labels.py --videos "
          f"{dst} --out_csv labels.csv")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile parse_ubi_fights_labels.py
"""
Build a labels.csv (same format evaluate_on_videos.py expects) directly
from UBI-Fights filenames - no manual labeling needed, since the label
is already encoded in the filename by the dataset's own convention:

    F_id_environment_camera_color.mp4   -> fight        (label 1)
    N_id_environment_camera_color.mp4   -> normal        (label 0)

This does not validate the labels against the actual video content -
it trusts the dataset's naming convention. That's a reasonable trust
level for an established benchmark dataset (unlike guessing labels by
hand), but if anything looks off during review (see
evaluate_on_videos.py's printed per-video predictions), spot-check a
few filenames against the actual footage before assuming the model is
wrong.

Usage:
    python parse_ubi_fights_labels.py --videos /path/to/ubi_fights --out_csv labels.csv
"""
import argparse
import csv
from pathlib import Path

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}


def label_from_filename(name):
    if name.startswith("F_"):
        return 1
    if name.startswith("N_"):
        return 0
    return None  # doesn't match the convention - don't guess


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--videos", required=True, help="Directory of UBI-Fights video files")
    parser.add_argument("--out_csv", default="labels.csv")
    args = parser.parse_args()

    video_dir = Path(args.videos)
    video_files = sorted(p for p in video_dir.iterdir() if p.suffix.lower() in VIDEO_EXTS)

    rows = []
    unmatched = []
    for vp in video_files:
        label = label_from_filename(vp.name)
        if label is None:
            unmatched.append(vp.name)
            continue
        rows.append((vp.name, label))

    with open(args.out_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["file", "label"])
        writer.writerows(rows)

    n_pos = sum(l for _, l in rows)
    print(f"Wrote {len(rows)} labels to {args.out_csv} ({n_pos} fight, {len(rows) - n_pos} normal)")
    if unmatched:
        print(f"\n{len(unmatched)} file(s) did not match the F_/N_ naming convention and were skipped:")
        for name in unmatched[:20]:
            print(f"  {name}")
        if len(unmatched) > 20:
            print(f"  ... and {len(unmatched) - 20} more")
        print("These should be checked manually rather than assumed safe to ignore.")


if __name__ == "__main__":
    main()


In [ ]:
import shutil
import subprocess
import os
from pathlib import Path

UBI_EVAL_DIR = Path("/content/data/ubi_fights_eval")
UBI_EVAL_DRIVE_BACKUP = "/content/drive/MyDrive/ubi_fights_eval_backup.zip"
UBI_LABELS_LOCAL = "/content/data/ubi_fights_labels.csv"
UBI_LABELS_DRIVE = "/content/drive/MyDrive/ubi_fights_labels.csv"

restored = False
if not (UBI_EVAL_DIR.exists() and any(UBI_EVAL_DIR.glob("*.mp4"))):
    backup = Path(UBI_EVAL_DRIVE_BACKUP)
    if backup.exists():
        print("Restoring UBI-Fights eval subsample from Drive backup...")
        UBI_EVAL_DIR.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(backup), "/tmp/ubi_eval_restore", "zip")
        for f in Path("/tmp/ubi_eval_restore").rglob("*.mp4"):
            shutil.copy2(f, UBI_EVAL_DIR / f.name)
        if Path(UBI_LABELS_DRIVE).exists():
            shutil.copy2(UBI_LABELS_DRIVE, UBI_LABELS_LOCAL)
        restored = any(UBI_EVAL_DIR.glob("*.mp4"))
        print(f"Restored {len(list(UBI_EVAL_DIR.glob('*.mp4')))} eval clips.")

if not restored:
    subprocess.run(["python", "subsample_ubi_fights.py", "--src", "/content/data/ubi_fights_raw",
                     "--dst", str(UBI_EVAL_DIR), "--n_per_class", "30"])
    subprocess.run(["python", "parse_ubi_fights_labels.py", "--videos", str(UBI_EVAL_DIR),
                     "--out_csv", UBI_LABELS_LOCAL])
    # Back up - small (60 clips), cheap, and keeps the same eval set
    # across sessions rather than relying on the random seed reproducing
    # it identically. Skipped safely (not silently to local storage) if
    # Drive isn't mounted.
    if os.path.ismount("/content/drive"):
        shutil.make_archive("/content/drive/MyDrive/ubi_fights_eval_backup", "zip", root_dir=str(UBI_EVAL_DIR))
        shutil.copy2(UBI_LABELS_LOCAL, UBI_LABELS_DRIVE)
        print("Backed up eval subsample + labels to Drive.")
    else:
        print("Google Drive not mounted - skipping eval-set backup. Run the "
              "Drive-mount cell first for this to persist.")

subprocess.run(["python", "evaluate_on_videos.py", "--videos", str(UBI_EVAL_DIR),
                 "--labels", UBI_LABELS_LOCAL, "--out_csv", "/content/data/ubi_fights_predictions.csv"])


### Target-domain check: hand-labeled patient/caregiver clips

`label_videos.py` is an interactive, resumable labeling helper - it does not
open the video (watch each clip in a normal player first), and it writes
each label to disk immediately so an interrupted session only loses the
clip in progress. It is written for a local terminal (watching a video while
answering a prompt) rather than for running unattended inside a Colab cell;
run it on a local machine against the folder of recorded/target-domain clips,
then bring the resulting `labels.csv` back here.

**Write the labeling rule down before labeling anything, and apply it
identically to every clip:**

> Label 1 only if there is aggressive physical contact (strike, push, slap,
> grab intended to harm/intimidate) between the two parties. Label 0 for calm
> interaction, verbal-only agitation, and caregiver restraint that is not
> itself aggressive.

Restraint-vs-aggression is the ambiguous case worth deciding explicitly up
front for this domain - a caregiver restraining an agitated patient involves
contact and looks superficially similar to aggression, but it isn't the thing
being detected.

In [ ]:
%%writefile label_videos.py
"""
Interactive hand-labeling helper for building labels.csv (used by
evaluate_on_videos.py --labels).

Before running: decide and write down the labeling rule in advance. For
patient/caregiver contact violence, something like:

    Label 1 only if there is aggressive physical contact (strike, push,
    slap, grab intended to harm/intimidate) between the two parties.
    Label 0 for calm interaction, verbal-only agitation, and caregiver
    restraint that is not itself aggressive.

Apply the same rule to every clip. If the rule gets written down partway
through labeling, go back and re-check the clips already done -
otherwise the first N labels and the rest were produced by different
criteria, a data-quality problem indistinguishable from noise once it's
in the CSV.

This script does not open the video (no reliable cross-platform GUI
launch, and this typically runs somewhere without a display attached to
the video files anyway). Watch each clip separately in a normal video
player, then come back and answer the prompt.

Resumable: progress is written to the output CSV after every single
label, so an interrupted session loses at most the clip in progress,
same resume pattern as extract_features.py's batch_run.
"""
import argparse
import csv
from pathlib import Path

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}


def load_existing(out_csv):
    done = {}
    p = Path(out_csv)
    if not p.exists():
        return done
    with open(p) as f:
        for row in csv.reader(f):
            if len(row) < 2:
                continue
            try:
                done[row[0]] = int(row[1])
            except ValueError:
                continue  # header row
    return done


def prompt_label(filename):
    while True:
        ans = input(f"{filename}  [1=violent/aggressive, 0=calm, s=skip/ambiguous, q=quit]: ").strip().lower()
        if ans in ("0", "1"):
            return int(ans)
        if ans == "s":
            return None
        if ans == "q":
            raise KeyboardInterrupt
        print("  invalid input, try again")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--videos", required=True, help="Directory of video files to label")
    parser.add_argument("--out_csv", default="labels.csv")
    args = parser.parse_args()

    video_dir = Path(args.videos)
    video_files = sorted(p for p in video_dir.iterdir() if p.suffix.lower() in VIDEO_EXTS)

    done = load_existing(args.out_csv)
    remaining = [p for p in video_files if p.name not in done]
    print(f"{len(video_files)} videos total, {len(done)} already labeled, {len(remaining)} remaining.\n")

    out_path = Path(args.out_csv)
    is_new = not out_path.exists()
    f = open(out_path, "a", newline="")
    writer = csv.writer(f)
    if is_new:
        writer.writerow(["file", "label"])

    try:
        for vp in remaining:
            label = prompt_label(vp.name)
            if label is None:
                print(f"  skipped {vp.name} (not written - treat as excluded, not as 0)")
                continue
            writer.writerow([vp.name, label])
            f.flush()
    except KeyboardInterrupt:
        print("\nStopped early. Progress saved.")
    finally:
        f.close()

    final = load_existing(args.out_csv)
    n_pos = sum(final.values())
    print(f"\n{len(final)} labeled so far ({n_pos} violent, {len(final) - n_pos} calm). Saved to {args.out_csv}")


if __name__ == "__main__":
    main()


In [ ]:
from pathlib import Path

# This step now runs automatically once crop_labeled_clips.py (see the
# "Fixing the cross-domain false-positive gap" section below) has been
# run with a filled-in manifest - it writes clips + labels_target.csv
# directly into TARGET_CLIPS, so both paths point at the same directory.
# For different hand-labeled footage instead (see label_videos.py
# above), copy/paste labels_target.csv into TARGET_CLIPS to match.

TARGET_CLIPS = Path("/content/data/target_domain_clips")
TARGET_LABELS = TARGET_CLIPS / "labels_target.csv"

if TARGET_CLIPS.exists() and TARGET_LABELS.exists():
    import subprocess
    result = subprocess.run(
        ["python", "evaluate_on_videos.py", "--videos", str(TARGET_CLIPS),
         "--labels", str(TARGET_LABELS), "--out_csv", "/content/data/target_domain_predictions.csv"],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
else:
    print("Target-domain clips/labels not found yet - skipping this step for now.\n"
          f"Expected clips at {TARGET_CLIPS} and labels at {TARGET_LABELS}.\n"
          "Fill in crop_manifest.csv and run crop_labeled_clips.py (below), "
          "or label footage manually with label_videos.py, then rerun this cell.")


### Reading the results

Report the RLVS test metrics, the UBI-Fights cross-domain accuracy, and the
target-domain accuracy as three separate numbers - do not average them or
treat any one as a stand-in for another; each answers a different question
(within-distribution generalization, camera-domain generalization,
population-domain generalization).

For both new eval runs, read the per-video predictions individually, not just
the aggregate accuracy. Specifically check for:

- `[no usable track]` lines - extraction failed on that video entirely
  (no person tracked for `MIN_SEQ_LEN` frames). This is a pipeline problem,
  not a model problem, and it stays silent unless checked for.
- Wrong predictions on the target-domain set in particular - given the
  physicality gap between RLVS's able-bodied fighters and a frail/restrained
  patient, false negatives (missed weak/low-amplitude strikes) are the
  expected failure mode to look for first.

## Fixing the cross-domain false-positive gap

Running `evaluate_on_videos.py` against a 30/30 balanced UBI-Fights sample
(real fixed-camera CCTV footage, unlike RLVS's handheld street video) showed
an asymmetric failure:

| | RLVS test set (original) | UBI-Fights eval sample |
|---|---|---|
| Violence recall | 0.79 | 0.83 (25/30) |
| Non-violence recall | 0.72 | 0.33 (7/21, +2 unusable tracks) |

Violence recall held up fine across domains. Non-violence recall collapsed --
the model calls ordinary CCTV footage "violence" roughly two-thirds of the
time. That drop (0.72 -> 0.33 on a 21-clip sample) is too large to be sampling
noise. The likely mechanism: the model never saw non-violence examples from a
fixed-camera domain during training, so it has no basis for learning that two
people close together with some motion can be calm in that setting - it may
be leaning on proximity/motion-magnitude as a shortcut rather than the actual
trajectory shape RLVS-only training was supposed to teach it.

**Before retraining, go watch a handful of the confident false positives**
(e.g. clips predicted violence at p>0.9 but labeled 0 in
`ubi_fights_predictions.csv`) to confirm this mechanism - hugging, playing,
or fast-but-benign motion tripping the same trigger as a real strike would
confirm it's a training-coverage problem, which is what the fix below
addresses. If it turns out to be something else entirely, the fix below won't
help and the failure needs a different diagnosis first.

**The fix required both more training coverage and a threshold
correction.** Adding calm cross-domain footage raised non-violence
recall substantially, but it also shifted the model's probability
distribution, so the default 0.5 threshold became badly calibrated for
this domain. Re-sweeping the operating point afterwards showed 0.30
improves accuracy and violence recall simultaneously - not a trade
between error types, but a correction of a mis-set decision boundary.
Both steps were necessary; neither alone was sufficient.

### Step 1: build an augmented feature set

Adds more UBI-Fights non-violence clips as label-0 training examples,
reusing the same `extract_features.py` pipeline (identical normalization,
boundary-clamp filtering, tracker) so features stay consistent with the
original RLVS-derived cache.

**Critically, this must exclude every clip already used in the 30/30 eval
sample.** Training on clips already used for evaluation invalidates the
before/after comparison the same way testing on training data would -
`build_augmented_dataset.py` checks filenames against `ubi_fights_eval`
explicitly and skips anything already there.

### Second calm-footage source: UCF-Crime, and additional target-domain clips

UCF-Crime adds a second, independent camera domain - real surveillance
footage. "Calm" needs to be learned across more than one camera style, not
just UBI-Fights'.

**Correction:** the first Kaggle mirror I used here (`odins0n/ucf-crime-dataset`)
turned out to be a pre-extracted 64x64 still-image version, not video --
confirmed via its own listing ("images extracted from every video... every
10th frame... 64x64 PNG"), which is incompatible with this pipeline (no
video file to track a person across, and 64x64 is too small for reliable
pose detection anyway). Switched to `bypktt/ucf-crimes`, which signals
"video classification" rather than "image" in its own metadata - more
likely to be actual video, but **not independently confirmed** the way
UBI-Fights was. The download cell below prints a breakdown of file
extensions it actually finds, specifically so a second wrong guess is
caught in seconds instead of after a long download.

`crop_labeled_clips.py` turns the 7 hand-reviewed UCLA timestamps into short
clips: the "argument" ones (verbal, no physical contact) go into training as
label 0, the real-violence ones go into eval only - 3 clips is too few to
train on without overfitting to their specifics, but enough for a small
target-domain sanity check. **`crop_manifest.csv` still needs to be filled
in manually** - that's the one part that genuinely requires information
only available locally (file paths and a judgment call on which clips are
real violence vs. argument).

In [ ]:
import subprocess
from pathlib import Path
from collections import Counter

UCF_RAW = Path("/content/data/ucf_crime_raw")

if UCF_RAW.exists() and any(UCF_RAW.rglob("*.mp4")):
    print(f"{UCF_RAW} already has videos, skipping download.")
else:
    result = subprocess.run(
        ["kaggle", "datasets", "download", "-d", "bypktt/ucf-crimes",
         "-p", str(UCF_RAW), "--unzip"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("UCF-Crime download failed (non-fatal - augmentation will just skip this source):")
        print(result.stderr)
    else:
        n_mp4 = len(list(UCF_RAW.rglob("*.mp4")))
        print(f"Downloaded UCF-Crime, found {n_mp4} .mp4 files.")

# Diagnostic: show what file types actually got downloaded, regardless of
# outcome above - catches a repeat of the "this mirror is actually images,
# not video" problem immediately instead of silently after the fact.
if UCF_RAW.exists():
    ext_counts = Counter(p.suffix.lower() for p in UCF_RAW.rglob("*") if p.is_file())
    print(f"\nFile types found under {UCF_RAW}: {dict(ext_counts.most_common(10))}")
    if not any(ext in ext_counts for ext in (".mp4", ".avi", ".mov", ".mkv")):
        print("warning: no video file extensions found - this mirror may not be "
              "video data either. Augmentation will skip this source gracefully "
              "(find_ucf_normal_clips returns empty), but it's not contributing "
              "anything useful. Check the folder contents manually before "
              "spending more time on this source.")


In [ ]:
%%writefile crop_labeled_clips.py
"""
Crop short clips out of longer source videos at given timestamps, and
route each one to either the training set (label 0, non-violence
"argument" clips) or the eval-only set (label 1, real physical
violence) based on a manually filled-in manifest CSV.

Why real-violence clips are eval-only, not training data: with only 3
of them, fine-tuning on them would just memorize those 3 clips'
specific quirks rather than learn anything generalizable - a small-
sample overfitting trap, not a real fix. They're still valuable as a
qualitative target-domain check, just not as training signal.

Why the label is manual, not model-generated: these clips were already
watched and classified as arguments vs real violence by a human, which
is real ground truth. Auto-labeling them with a pretrained detector
would replace a verified judgment with a guess from a model that could
easily carry the same bias this whole augmentation effort is trying to
fix.

Manifest format (crop_manifest.csv):
    video_path,start,end,label,split
    /content/data/dementia_videos/aggressive_language.mp4,0:55,1:28,?,?
    ...
start/end accept M:SS or seconds. label is 0 or 1, filled in based on
what was determined while watching. split is "train" or "eval" -
0-labeled clips normally go to "train", 1-labeled clips normally go to
"eval" given there are only 3, but the column is explicit rather than
inferred so it can be overridden.

Requires ffmpeg (already present in the Colab base image).
"""
import argparse
import csv
import shutil
import subprocess
from pathlib import Path


def parse_timestamp(ts: str) -> float:
    ts = ts.strip()
    if ":" in ts:
        parts = [float(p) for p in ts.split(":")]
        seconds = 0.0
        for p in parts:
            seconds = seconds * 60 + p
        return seconds
    return float(ts)


def crop_clip(video_path: str, start: str, end: str, out_path: Path):
    start_s = parse_timestamp(start)
    end_s = parse_timestamp(end)
    duration = end_s - start_s
    if duration <= 0:
        raise ValueError(f"end ({end}) must be after start ({start}) for {video_path}")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    # -ss before -i is fast (seek), re-encoding (-c:v libx264) rather than
    # -c copy because stream-copy crops can only cut on keyframes, which
    # would make the actual clipped range inaccurate for short clips.
    cmd = [
        "ffmpeg", "-y", "-ss", str(start_s), "-i", str(video_path), "-t", str(duration),
        "-c:v", "libx264", "-c:a", "aac", "-loglevel", "error", str(out_path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed on {video_path} [{start}-{end}]: {result.stderr}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--manifest", required=True, help="CSV: video_path,start,end,label,split")
    parser.add_argument("--train_out", default="/content/data/target_domain_train")
    parser.add_argument("--eval_out", default="/content/data/target_domain_clips")
    args = parser.parse_args()

    train_out = Path(args.train_out)
    eval_out = Path(args.eval_out)
    eval_labels = {}

    with open(args.manifest) as f:
        rows = list(csv.DictReader(f))

    if not rows:
        print(f"{args.manifest} is empty - nothing to crop.")
        return

    n_cropped = 0
    for i, row in enumerate(rows, 1):
        video_path = row["video_path"].strip()
        label = row["label"].strip()
        split = row["split"].strip().lower()

        if not video_path or label not in ("0", "1") or split not in ("train", "eval"):
            print(f"[{i}/{len(rows)}] skipped - incomplete row (fill in video_path/label/split): {row}")
            continue
        if not Path(video_path).exists():
            print(f"[{i}/{len(rows)}] skipped - file not found: {video_path}")
            continue

        clip_name = f"{Path(video_path).stem}_{row['start'].replace(':', '')}_{row['end'].replace(':', '')}.mp4"

        if split == "train":
            if label != "0":
                print(f"[{i}/{len(rows)}] warning: label=1 clip routed to train - "
                      f"with only a handful of real-violence clips this will likely overfit "
                      f"to their specifics rather than generalize. Consider split=eval instead.")
            out_path = train_out / clip_name
        else:
            out_path = eval_out / clip_name
            eval_labels[clip_name] = int(label)

        try:
            crop_clip(video_path, row["start"], row["end"], out_path)
            print(f"[{i}/{len(rows)}] cropped -> {out_path} (label={label}, split={split})")
            n_cropped += 1
        except Exception as e:
            print(f"[{i}/{len(rows)}] failed: {e}")

    if eval_labels:
        eval_out.mkdir(parents=True, exist_ok=True)
        labels_csv = eval_out / "labels_target.csv"
        with open(labels_csv, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["file", "label"])
            for fname, label in eval_labels.items():
                w.writerow([fname, label])
        print(f"\nWrote {labels_csv} for evaluate_on_videos.py --labels")

    print(f"\n{n_cropped}/{len(rows)} clips cropped successfully.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile crop_manifest.csv
video_path,start,end,label,split
,0:55,1:28,,
,1:06,1:20,,
,0:42,0:52,,
,0:40,1:14,,
,1:03,2:10,,
,0:44,1:00,,
,0:56,1:06,,


**Fill in `crop_manifest.csv` before running the next cell** - open it in the
Colab file browser, add the `video_path` for each of the 7 rows (wherever
the UCLA video files were saved/uploaded in this session), and the `label`
(0 = argument, 1 = real violence) and `split` (train/eval) based on the
judgment already made while watching them. Only 3 of the 7 should be
label=1/split=eval.

In [ ]:
subprocess.run(["python", "crop_labeled_clips.py", "--manifest", "crop_manifest.csv",
                 "--train_out", "/content/data/target_domain_train",
                 "--eval_out", "/content/data/target_domain_clips"])


In [ ]:
from pathlib import Path

# Automatic re-check now that crop_labeled_clips.py has run - the
# target-domain eval cell much earlier in this notebook (right after
# label_videos.py) runs before this crop step exists in the linear
# top-to-bottom order, so on a first full run it always finds nothing and
# reports "not found yet, skipping" - correctly, but pointlessly, since
# the data it needs doesn't exist until right now. This duplicates that
# same check here, in the position where it can actually succeed on a
# single top-to-bottom run instead of requiring a manual scroll back up
# to rerun that earlier cell.
TARGET_CLIPS = Path("/content/data/target_domain_clips")
TARGET_LABELS = TARGET_CLIPS / "labels_target.csv"

if TARGET_CLIPS.exists() and TARGET_LABELS.exists():
    result = subprocess.run(
        ["python", "evaluate_on_videos.py", "--videos", str(TARGET_CLIPS),
         "--labels", str(TARGET_LABELS), "--out_csv", "/content/data/target_domain_predictions.csv"],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
else:
    print("crop_manifest.csv was empty or incomplete - no target-domain eval clips "
          "produced. Fill it in and rerun the two cells above, then this one.")


In [ ]:
%%writefile build_augmented_dataset.py
"""
Build an augmented feature-cache directory: the original RLVS feature
set plus new cross-domain non-violence clips from UBI-Fights and
UCF-Crime, plus optionally extra manually-cropped domain-specific
clips (see crop_labeled_clips.py).

Why this exists:
evaluate_on_videos.py showed the trained model has strong recall on
cross-domain violence but very poor recall on cross-domain non-violence
- it over-predicts "violence" on ordinary CCTV footage it wasn't
trained on. The fix is more training coverage of calm cross-domain
footage from multiple camera domains, not a threshold tweak.

Important: clips used for evaluation must never be used for training.
Every filename already in the UBI-Fights eval directory is excluded.

Random sampling, not a sorted slice: clips are chosen with
random.sample (fixed seed), not all_normal[:n] - a sorted alphabetical
slice can accidentally grab a whole cluster of clips from one
camera/session, which is the likely reason an earlier run of this
script saw a near-total failure streak that a randomly-sampled eval set
never showed.

UCF-Crime normal clips: a second, independent camera domain, added for
the same reason UBI-Fights was added - "calm" needs to be learned
across multiple camera styles, not just one. UCF-Crime's Kaggle mirror
layout hasn't been directly inspected, so normal-video detection here
is a heuristic (path/filename contains "normal", case-insensitive)
rather than a hardcoded folder path. If this finds zero or a
suspiciously small number of clips, print what folders actually exist
under ucf_raw_dir and adjust find_ucf_normal_clips.

Visibility, resumability, Drive backup: see inline comments - same
patterns as before (progress printed per clip, safe to interrupt and
rerun, periodic + final backup to Drive so a disconnect costs at most
one backup interval, not the whole run).
"""
import argparse
import random
import re
import shutil
import tempfile
from pathlib import Path

from extract_features import process_clip

DRIVE_BACKUP_ZIP = "/content/drive/MyDrive/features_v2_augmented_backup.zip"
BACKUP_EVERY = 25


def restore_from_drive(out_path: Path, backup_zip: str = DRIVE_BACKUP_ZIP):
    if any(out_path.glob("*.npz")):
        return
    backup = Path(backup_zip)
    if not backup.exists():
        print(f"No Drive backup found at {backup_zip} - starting fresh.")
        return
    print(f"Restoring {out_path} from Drive backup...")
    with tempfile.TemporaryDirectory() as tmp:
        shutil.unpack_archive(str(backup), tmp, "zip")
        found_npz = list(Path(tmp).rglob("*.npz"))
        out_path.mkdir(parents=True, exist_ok=True)
        for f in found_npz:
            shutil.copy2(f, out_path / f.name)
        log_matches = list(Path(tmp).rglob("augment_log.txt"))
        if log_matches:
            shutil.copy2(log_matches[0], out_path / "augment_log.txt")
        print(f"Restored {len(found_npz)} feature files from Drive backup.")


def _assert_drive_mounted(path_str):
    import os
    if str(path_str).startswith("/content/drive") and not os.path.ismount("/content/drive"):
        raise RuntimeError(
            f"Google Drive is not mounted at /content/drive (tried to use {path_str}). "
            "Run the Drive-mount cell first - otherwise this backup would silently "
            "write to local, non-persistent storage instead of Drive."
        )


def backup_to_drive(out_path: Path, backup_zip: str = DRIVE_BACKUP_ZIP):
    _assert_drive_mounted(backup_zip)
    # See the matching fix in extract_features.py's _backup_features_to_drive:
    # make_archive appends its own extension to the base name it's given and
    # returns the real path it wrote, so use that instead of reconstructing
    # the filename via string replacement (the old ".zip"-substring-strip
    # approach produced a name that never matched what was actually created,
    # crashing shutil.move with FileNotFoundError on every backup attempt).
    base_name = backup_zip[:-4] if backup_zip.endswith(".zip") else backup_zip
    archive_path = shutil.make_archive(base_name + ".tmp", "zip", root_dir=str(out_path))
    shutil.move(archive_path, backup_zip)
    print(f"Backed up {out_path} to {backup_zip}")


def find_ucf_normal_clips(ucf_raw_dir):
    video_exts = {".mp4", ".avi", ".mov", ".mkv"}
    return sorted(
        p for p in Path(ucf_raw_dir).rglob("*")
        if p.suffix.lower() in video_exts and "normal" in str(p).lower()
    )


def _process_batch(chosen, out_dir, out_path, log_path, attempted, source_label):
    """Shared processing loop for any list of (Path) video clips, all
    labeled 0 (non-violence). source_label is just for the printed
    progress lines, so UBI-Fights and UCF-Crime clips are distinguishable
    in the console output."""
    wrote = 0
    to_process = [vp for vp in chosen if vp.stem not in attempted]
    print(f"[{source_label}] {len(chosen) - len(to_process)} of {len(chosen)} already attempted, "
          f"{len(to_process)} left to process.\n")

    for i, vp in enumerate(to_process, 1):
        ok = process_clip(str(vp), label=0, out_dir=out_dir)
        with open(log_path, "a") as f:
            f.write(vp.stem + "\n")
        attempted.add(vp.stem)
        wrote += int(ok)
        status = "ok" if ok else "empty (no track reached MIN_SEQ_LEN)"
        print(f"[{source_label} {i}/{len(to_process)}] {vp.name}: {status}")

        if i % BACKUP_EVERY == 0:
            backup_to_drive(out_path)

    if to_process:
        backup_to_drive(out_path)

    return wrote, len(chosen) - len(to_process)


def build_augmented_features(rlvs_features_dir, ubi_raw_dir, ubi_eval_dir,
                              out_dir, n_new_normal=250, seed=42,
                              ucf_raw_dir=None, n_ucf_normal=150,
                              extra_normal_dir=None):
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    restore_from_drive(out_path)

    # --- RLVS base ---
    existing = list(Path(rlvs_features_dir).glob("*.npz"))
    already_in_out = {p.name for p in out_path.glob("*.npz")}
    missing = [f for f in existing if f.name not in already_in_out]
    if missing:
        print(f"Copying {len(missing)} RLVS feature files into {out_dir} "
              f"({len(existing) - len(missing)} already present)...")
        for f in missing:
            shutil.copy2(f, out_path / f.name)
    else:
        print(f"All {len(existing)} RLVS feature files already present in {out_dir}.")

    log_path = out_path / "augment_log.txt"
    attempted = set(log_path.read_text().splitlines()) if log_path.exists() else set()
    for f in out_path.glob("*_id*.npz"):
        m = re.match(r"^(.*)_id\d+\.npz$", f.name)
        if m:
            attempted.add(m.group(1))

    # --- UBI-Fights normal (random sample, excluding eval clips) ---
    eval_names = {p.name for p in Path(ubi_eval_dir).iterdir()}
    all_ubi_normal = [p for p in Path(ubi_raw_dir).rglob("N_*.mp4") if p.name not in eval_names]
    print(f"{len(all_ubi_normal)} UBI-Fights normal clips available after excluding "
          f"{len(eval_names)} already-evaluated clips.")
    if len(all_ubi_normal) < n_new_normal:
        print(f"warning: requested {n_new_normal} but only {len(all_ubi_normal)} available; using all of them.")
        ubi_chosen = all_ubi_normal
    else:
        rng = random.Random(seed)
        ubi_chosen = rng.sample(all_ubi_normal, n_new_normal)

    ubi_wrote, ubi_skipped = _process_batch(ubi_chosen, out_dir, out_path, log_path, attempted, "UBI")

    # --- UCF-Crime normal (second camera domain) ---
    ucf_wrote, ucf_skipped = 0, 0
    if ucf_raw_dir:
        all_ucf_normal = find_ucf_normal_clips(ucf_raw_dir)
        print(f"\n{len(all_ucf_normal)} UCF-Crime normal clips found under {ucf_raw_dir}.")
        if not all_ucf_normal:
            print(f"warning: found zero - the Kaggle mirror's folder layout may not match the "
                  f"'normal' substring heuristic. List the contents of {ucf_raw_dir} and adjust "
                  f"find_ucf_normal_clips() if needed.")
        else:
            if len(all_ucf_normal) < n_ucf_normal:
                print(f"warning: requested {n_ucf_normal} but only {len(all_ucf_normal)} available; using all.")
                ucf_chosen = all_ucf_normal
            else:
                rng2 = random.Random(seed + 1)  # different seed, independent sample
                ucf_chosen = rng2.sample(all_ucf_normal, n_ucf_normal)
            ucf_wrote, ucf_skipped = _process_batch(ucf_chosen, out_dir, out_path, log_path, attempted, "UCF")

    # --- Extra cropped domain-specific clips (crop_labeled_clips.py's
    # train_out folder) - already isolated, single-source, all label 0,
    # so no sampling/heuristic needed, just process every clip in it.
    extra_wrote, extra_skipped = 0, 0
    if extra_normal_dir:
        extra_clips = sorted(p for p in Path(extra_normal_dir).glob("*.mp4"))
        print(f"\n{len(extra_clips)} domain-specific clips found under {extra_normal_dir}.")
        if extra_clips:
            extra_wrote, extra_skipped = _process_batch(extra_clips, out_dir, out_path, log_path, attempted, "TARGET")

    total_npz = len(list(out_path.glob("*.npz")))
    print(f"\n=== Summary ===")
    print(f"UBI-Fights: {ubi_wrote} newly processed usable, {ubi_skipped} already done")
    if ucf_raw_dir:
        print(f"UCF-Crime:  {ucf_wrote} newly processed usable, {ucf_skipped} already done")
    if extra_normal_dir:
        print(f"Target-domain (manual clips): {extra_wrote} newly processed usable, {extra_skipped} already done")
    print(f"Total feature files in {out_dir}: {total_npz}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--rlvs_features", default="/content/features_v2")
    parser.add_argument("--ubi_raw", required=True)
    parser.add_argument("--ubi_eval", required=True)
    parser.add_argument("--out_dir", required=True)
    parser.add_argument("--n_new_normal", type=int, default=250)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--ucf_raw", default=None, help="UCF-Crime raw dataset dir (optional second calm-footage source)")
    parser.add_argument("--n_ucf_normal", type=int, default=150)
    parser.add_argument("--extra_normal_dir", default=None, help="Extra cropped domain-specific label-0 clips")
    args = parser.parse_args()
    build_augmented_features(args.rlvs_features, args.ubi_raw, args.ubi_eval,
                              args.out_dir, args.n_new_normal, args.seed,
                              args.ucf_raw, args.n_ucf_normal, args.extra_normal_dir)


if __name__ == "__main__":
    main()


In [ ]:
from build_augmented_dataset import build_augmented_features

build_augmented_features(
    rlvs_features_dir="/content/features_v2",
    ubi_raw_dir="/content/data/ubi_fights_raw",
    ubi_eval_dir="/content/data/ubi_fights_eval",
    out_dir="/content/features_v2_augmented",
    n_new_normal=754,
    ucf_raw_dir=None,
    extra_normal_dir=None,
)

### Step 2: fine-tune from the existing best checkpoint

Continues training from `lstm_stage1_best.pt` at a lower learning rate on the
combined dataset, rather than training from scratch - cheaper, and starts
from weights that already work well on violence recall. Saved under a new
name (`lstm_stage2_augmented_best.pt`) so the original stage-1 checkpoint and
its RLVS test results are never overwritten.

Watch the class balance: adding ~300 label-0 clips on top of the existing
RLVS split shifts the ratio further toward non-violence. If violence recall
drops noticeably in the fine-tuned test metrics below, that's the likely
cause, and the new-clip count or a class weight would need adjusting.

In [ ]:
from pathlib import Path
print(len(list(Path("/content/features_v2_augmented").glob("*.npz"))), "feature files")

In [ ]:
%%writefile fine_tune.py
"""
Fine-tune the fight-detection LSTM on the augmented dataset (RLVS +
cross-domain non-violence clips), starting from the existing best
checkpoint instead of training from scratch.

Why not just call train_stage1.train() again:
train() resumes from CHECKPOINT_PATH (the per-epoch resumable
checkpoint), which by now sits at the final epoch of the already-
completed original run. Calling it again either does nothing (already
past PATIENCE) or resumes a training schedule that doesn't correspond
to "start a new short run on new data from the best known weights."
Fine-tuning needs its own entry point: load the best checkpoint's
weights only (not its optimizer state or epoch count), train briefly
at a lower learning rate, and save under a new name so the original
stage1 checkpoint and results are never overwritten.

Why the split is recomputed here:
three_way_split groups by source_clip so a clip's tracks never cross
splits. The new UBI-Fights clips were never part of the original split,
so the split has to be recomputed over the full combined npz directory
- reusing the old split object would silently drop the new data from
having any test/val representation.

This does not replace evaluating on the original 30/30 UBI-Fights eval
set with evaluate_on_videos.py after fine-tuning - that comparison
(same eval clips, before vs. after checkpoint) is the one that actually
answers whether the false-positive problem improved. This script's own
test-split evaluation is a useful sanity check but mixes RLVS and new
UBI-Fights clips together, so it isn't a clean apples-to-apples
comparison against the original RLVS-only test number.
"""
import argparse
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from train_stage1 import (
    MotionLSTM, KeypointSeqDataset, three_way_split, evaluate,
    evaluate_test_full, DEVICE, BATCH_SIZE, PATIENCE
)


def fine_tune(npz_dir, base_checkpoint, save_prefix="stage2_augmented",
              epochs=15, lr=1e-4):
    train_files, val_files, test_files = three_way_split(npz_dir)
    print(f"Train: {len(train_files)}  Val: {len(val_files)}  Test: {len(test_files)}  "
          f"(recomputed split over combined RLVS + UBI-Fights-normal data)")

    train_loader = DataLoader(KeypointSeqDataset(train_files), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(KeypointSeqDataset(val_files), batch_size=BATCH_SIZE)
    test_loader = DataLoader(KeypointSeqDataset(test_files), batch_size=BATCH_SIZE)

    model = MotionLSTM().to(DEVICE)
    ckpt = torch.load(base_checkpoint, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    print(f"Loaded base weights from {base_checkpoint} (val_acc={ckpt.get('val_acc', 'n/a')})")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    epochs_without_improvement = 0
    best_path = Path(base_checkpoint).parent / f"lstm_{save_prefix}_best.pt"

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for x, lengths, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x, lengths)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)

        avg_loss = total_loss / max(len(train_files), 1)
        val_acc, val_prec, val_rec = evaluate(model, val_loader)
        print(f"Epoch {epoch+1}/{epochs} | loss {avg_loss:.4f} | "
              f"val_acc {val_acc:.4f} | val_precision {val_prec:.4f} | val_recall {val_rec:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_without_improvement = 0
            torch.save({"epoch": epoch + 1, "model_state": model.state_dict(),
                        "val_acc": val_acc}, best_path)
            print(f"  -> new best (val_acc={val_acc:.4f}), saved to {best_path.name}")
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print(f"\nNo improvement for {PATIENCE} epochs, stopping early at epoch {epoch+1}.")
            break

    print(f"\nFine-tuning complete. Best val acc: {best_val_acc:.4f}")

    best_ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(best_ckpt["model_state"])
    evaluate_test_full(model, test_loader, save_prefix=save_prefix)

    print(f"\nFine-tuned checkpoint saved to: {best_path}")
    print("Next: rerun evaluate_on_videos.py against the original 30/30 UBI-Fights "
          f"eval set with --checkpoint {best_path} and compare against the earlier run "
          "- that comparison, not this function's own test split, is what answers "
          "whether the false-positive problem actually improved.")
    return model


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--npz_dir", required=True, help="Combined RLVS + new UBI-Fights-normal feature directory")
    parser.add_argument("--base_checkpoint", required=True, help="Path to lstm_stage1_best.pt")
    parser.add_argument("--save_prefix", default="stage2_augmented")
    parser.add_argument("--epochs", type=int, default=15)
    parser.add_argument("--lr", type=float, default=1e-4)
    args = parser.parse_args()
    fine_tune(args.npz_dir, args.base_checkpoint, args.save_prefix, args.epochs, args.lr)


if __name__ == "__main__":
    main()


In [ ]:
from pathlib import Path
import torch
from fine_tune import fine_tune
from train_stage1 import MotionLSTM, DEVICE

# NEW checkpoint name (stage2b, not stage2) - starting fresh from stage1
# with the bigger, multi-source, properly-randomized augmented dataset.
# Keeping the old lstm_stage2_augmented_best.pt around allows comparing
# both attempts rather than silently overwriting the first one.
STAGE2B_CKPT = Path("/content/drive/MyDrive/fight_detection_checkpoints/lstm_stage2b_augmented_best.pt")

if STAGE2B_CKPT.exists():
    print(f"Found existing checkpoint at {STAGE2B_CKPT}, skipping fine-tuning and loading it instead.")
    model = MotionLSTM().to(DEVICE)
    ckpt = torch.load(STAGE2B_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    print(f"Loaded (val_acc={ckpt.get('val_acc', 'n/a')}). "
          "Delete this checkpoint file first to fine-tune again from scratch.")
else:
    model = fine_tune(
        npz_dir="/content/features_v2_augmented",
        base_checkpoint="/content/drive/MyDrive/fight_detection_checkpoints/lstm_stage1_best.pt",
        save_prefix="stage2b_augmented",
        epochs=30,
        lr=1e-4,
    )


### Step 3: re-check on the untouched 30/30 UBI-Fights eval set

This is the comparison that actually answers whether the fix worked: same 60
clips as the original run, same script, only the checkpoint changes. The
fine-tuned model's own held-out test metrics (printed above) mix RLVS and new
UBI-Fights clips together and are a useful sanity check, but they are not a
clean substitute for this - this rerun is the one to trust.

In [ ]:
!python evaluate_on_videos.py --videos /content/data/ubi_fights_eval \
    --labels /content/data/ubi_fights_labels.csv \
    --checkpoint /content/drive/MyDrive/fight_detection_checkpoints/lstm_stage2b_augmented_best.pt \
    --out_csv /content/data/ubi_fights_predictions_stage2b.csv


In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score
df = pd.read_csv("/content/data/ubi_fights_predictions_stage2b.csv").dropna(subset=["label","max_prob"]).astype({"max_prob":float,"label":int})
print(len(df), "usable |  AUC:", round(roc_auc_score(df["label"], df["max_prob"]),4))
for t in [0.30,0.35,0.40,0.45,0.50]:
    p=(df["max_prob"]>=t).astype(int)
    print(f"thr={t:.2f} acc={(p==df['label']).mean():.3f} "
          f"NV={((p==0)&(df['label']==0)).sum()/(df['label']==0).sum():.3f} "
          f"V={((p==1)&(df['label']==1)).sum()/(df['label']==1).sum():.3f}")

Compare `ubi_fights_predictions_stage2b.csv` against the original
`ubi_fights_predictions.csv` directly: non-violence recall should move up
from 0.33 toward something closer to the RLVS baseline (0.72), and violence
recall should not have dropped meaningfully from 0.83. If non-violence
recall improved without violence recall regressing, move on to the
target-domain (patient/caregiver) hand-labeled check next. If it didn't
improve, the problem is deeper than training-data coverage and the false
positives need to be re-diagnosed (recheck the specific clips from Step 0,
looking for a pattern the added data didn't cover) before trying another
fix.

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score

df = pd.read_csv("/content/data/ubi_fights_predictions_stage2b.csv")
df = df.dropna(subset=["label", "max_prob"]).astype({"max_prob": float, "label": int})
print(f"{len(df)} usable rows")
print("cross-domain ROC-AUC:", round(roc_auc_score(df["label"], df["max_prob"]), 4))

for t in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    pred = (df["max_prob"] >= t).astype(int)
    acc = (pred == df["label"]).mean()
    nv = ((pred == 0) & (df["label"] == 0)).sum() / (df["label"] == 0).sum()
    v  = ((pred == 1) & (df["label"] == 1)).sum() / (df["label"] == 1).sum()
    print(f"thr={t:.2f}  acc={acc:.3f}  NV={nv:.3f}  V={v:.3f}")

In [ ]:
for t in [0.15, 0.20, 0.25, 0.30]:
    pred = (df["max_prob"] >= t).astype(int)
    acc = (pred == df["label"]).mean()
    nv = ((pred == 0) & (df["label"] == 0)).sum() / (df["label"] == 0).sum()
    v  = ((pred == 1) & (df["label"] == 1)).sum() / (df["label"] == 1).sum()
    print(f"thr={t:.2f}  acc={acc:.3f}  NV={nv:.3f}  V={v:.3f}")

## Frame-level violence detection with bounding boxes (separate method)

Everything above is the pose-based LSTM pipeline: track a person's skeleton
over time, classify the motion trajectory. This section is a completely
different, independent method - a pretrained YOLOv8 object detector
(Musawer14/fight_detection_yolov8 from Hugging Face) that classifies
individual frames directly, with a "Violence/Fight" class, plus optical-flow
motion gating so a static pose doesn't get flagged just because someone is
mid-punch-looking-pose without actual motion.

**Why this is useful alongside the LSTM pipeline, not a replacement for it:**
it doesn't require pose extraction or tracking at all, so it's a fast way to
scan a longer, unlabeled video (like the UCLA dementia training videos) and
see where in the video a detector thinks something violent is happening -
useful for finding the timestamp to trim into a short clip, without watching
the whole thing start to finish. It draws the actual bounding boxes it
detected on each flagged frame and saves them as images, so the output can
be visually confirmed (or rejected) rather than trusted as a bare confidence
number.

**What it deliberately does not do:** report real precision/recall (there's
no ground truth to score against unless the answer is already known), or
constitute a validated action-recognition model - optical flow motion
magnitude fires on any fast motion (running, dancing, sports), not
specifically fighting. Its output is a fast triage tool for finding
candidate moments to review manually, not a verified detector.

**Security note:** one of the two released checkpoints
(`yolo_small_weights.pt`) is flagged "Unsafe" by Hugging Face's own pickle
scanner, because its pickle stream references a mechanism capable of
reconstructing arbitrary Python objects on load - a known code-execution
vector. This script disassembles that checkpoint's pickle stream and checks
every referenced symbol against an allow-list before loading it, refusing
automatically if anything unexpected shows up. This is a heuristic static
check, not sandboxed execution - passing `--skip-small` when the small
model isn't needed loads only the Safe-scanned nano checkpoint.

In [ ]:
%%writefile extract_violent_frames.py
"""
Fight/violence detection pipeline using Musawer14/fight_detection_yolov8 (YOLOv8).
https://huggingface.co/Musawer14/fight_detection_yolov8

What this does
    1. Samples consecutive-frame "bursts" spread across the video (default:
       6 bursts of 5 consecutive frames = 30 frames total), rather than 30
       single frames spread evenly across the whole clip. Motion signals
       need temporally adjacent frames - single frames sampled seconds
       apart from a multi-minute video make optical flow meaningless.
    2. Runs both released YOLO checkpoints (nano + small) on every frame
       for a per-frame confidence score (the same detector as before).
    3. Computes dense optical flow (Farneback) between consecutive frames
       within each burst, as an independent motion-magnitude signal.
    4. Reports both a raw per-frame verdict (YOLO confidence alone, same
       as the original script) and, once --motion-threshold is supplied,
       a motion-gated verdict (YOLO confidence and local motion both
       above threshold), to compare whether requiring motion changes the
       outcome.

What this deliberately does not do
    It does not report "accuracy" in the strict sense (precision/recall
    against ground truth). Unless the video has a known true per-frame
    label, there's nothing to score detections against. The output is
    observational: what the model flagged, and how confidently, not a
    verified correctness metric.

    It also does not turn this into a real action-recognition model. Dense
    optical flow magnitude is a generic "how much motion is happening"
    signal - it fires on running, dancing, sports, or any fast motion,
    not specifically fighting. It's a cheap filter against static-pose
    false positives, not learned understanding of what a fight looks like.
    A proper fix for that is a temporal model (I3D/X3D/VideoMAE) trained on
    a labeled violence dataset (RWF-2000, Hockey Fight) - a different,
    larger project than this script.

Setup (run once, on a machine with normal internet - not a locked-down
sandbox, torch is 500MB+):
    pip install ultralytics opencv-python huggingface_hub

Weights
    Download both checkpoints from the HF repo's Files tab and place them
    next to this script (or point --weights-dir elsewhere):
        Yolo_nano_weights.pt
        yolo_small_weights.pt
    Or let this script auto-download them via huggingface_hub (see
    ensure_weights() below) - pass --download.

Security note on yolo_small_weights.pt
    Hugging Face's own pickle scanner flags this file "Unsafe" because its
    pickle stream references dill._dill._load_type, a mechanism capable of
    reconstructing arbitrary Python type objects during unpickling - a
    known code-execution vector, distinct from a checkpoint that only
    contains tensors. Yolo_nano_weights.pt scans "Safe" and does not have
    this issue.

    Before this script will load yolo_small_weights.pt, it disassembles the
    pickle stream (inspect_pickle_safety) and checks every referenced
    global/module name against an allow-list of expected torch/ultralytics
    symbols. This is a heuristic static check, not a sandboxed execution -
    it catches "this pickle references os.system" but cannot prove absence
    of cleverly-obfuscated behavior. Skipping the small model entirely
    (run nano only) avoids the question altogether.
"""

from __future__ import annotations

import argparse
import pickletools
import sys
import zipfile
import cv2
import numpy as np
from ultralytics import YOLO

from pathlib import Path


FIGHT_CLASS_ID = 1  # per model card: class 1 = Violence/Fight
DEFAULT_CONF = 0.25  # ultralytics default confidence threshold

# Substrings that are fine to see referenced inside a YOLO checkpoint pickle.
SAFE_PREFIXES = (
    "torch.",
    "ultralytics.",
    "collections.",
    "numpy.",
    "builtins.set",
    "__builtin__.set",
    "dill._dill._load_type",
)

# Substrings that should never legitimately appear in a model checkpoint's
# pickled object graph. Presence of any of these is a hard stop.
DANGEROUS_SUBSTRINGS = (
    "os.system", "os.popen", "subprocess", "eval", "exec(",
    "__import__", "socket", "shutil.rmtree", "posix.system",
)


def inspect_pickle_safety(weights_path: Path) -> list[str]:
    """
    Disassemble the pickle stream inside a YOLO .pt checkpoint (a zip
    archive containing data.pkl + raw tensor storages) and flag every
    GLOBAL/STACK_GLOBAL reference (i.e. every class/function the pickle
    will import and instantiate on load) that isn't in SAFE_PREFIXES, plus
    any string operand matching a known-dangerous substring.

    Two pickle opcode forms have to be handled separately:
      - GLOBAL (protocol 0-2): carries its own arg as "module name"
        (space-joined, NOT dot-joined - a first version of this function
        assumed dots and silently let every classic-protocol reference
        through unchecked).
      - STACK_GLOBAL (protocol 4+): carries no arg at all; the module and
        name are the two preceding string values pushed onto the pickle
        VM's stack, popped by this opcode.

    Returns a list of flagged strings. Empty list means nothing suspicious
    found (heuristic static check, not sandboxed execution - see module
    docstring for limits).
    """
    with zipfile.ZipFile(weights_path) as zf:
        pkl_names = [n for n in zf.namelist() if n.endswith("data.pkl")]
        if not pkl_names:
            raise RuntimeError(f"No data.pkl found inside {weights_path} - not a standard torch checkpoint zip.")
        data = zf.read(pkl_names[0])

    flagged: list[str] = []
    recent_strings: list[str] = []  # rolling window of the last 2 string pushes, for STACK_GLOBAL

    def check(target: str) -> None:
        lower = target.lower()
        if any(bad in lower for bad in DANGEROUS_SUBSTRINGS):
            flagged.append(f"[DANGEROUS] {target}")
            return
        if not any(target.startswith(p) or target == p.rstrip(".") for p in SAFE_PREFIXES):
            flagged.append(target)

    for opcode, arg, _pos in pickletools.genops(data):
        if opcode.name in ("SHORT_BINUNICODE", "BINUNICODE", "UNICODE",
                            "SHORT_BINSTRING", "BINSTRING") and isinstance(arg, str):
            recent_strings.append(arg)
            if len(recent_strings) > 2:
                recent_strings.pop(0)

        if opcode.name == "GLOBAL" and isinstance(arg, str):
            parts = arg.split(" ", 1)
            if len(parts) == 2:
                check(f"{parts[0]}.{parts[1]}")
        elif opcode.name == "STACK_GLOBAL":
            if len(recent_strings) >= 2:
                module, name = recent_strings[-2], recent_strings[-1]
                check(f"{module}.{name}")

    return flagged


def sample_frame_bursts(video_path: Path, total_frames: int = 30, burst_size: int = 5):
    """
    Sample consecutive-frame bursts spread across the video, instead of
    single frames spread evenly across the whole clip.

    Why: optical flow (or any motion signal) between two frames requires
    them to be temporally adjacent. Sampling single frames evenly across a
    multi-minute video - the original approach in this script - puts
    seconds of gap between "consecutive" samples; computing flow across
    that gap is meaningless, the scene may have changed entirely.

    Returns a list of bursts; each burst is a list of (frame_idx, frame)
    tuples of `burst_size` temporally consecutive frames. Burst start
    points are spread evenly across the video so the full clip is still
    covered, not just its first few seconds.
    """

    if total_frames % burst_size != 0:
        raise ValueError(
            f"--total-frames ({total_frames}) must be a multiple of --burst-size ({burst_size})."
        )
    num_bursts = total_frames // burst_size

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < total_frames:
        cap.release()
        raise RuntimeError(
            f"Video only has {total} decodable frames; need at least {total_frames}. "
            f"Use a longer clip, lower --total-frames, or lower --burst-size."
        )

    last_valid_start = max(total - burst_size, 0)
    denom = max(num_bursts - 1, 1)
    starts = sorted(set(
        min(int(i * last_valid_start / denom), last_valid_start)
        for i in range(num_bursts)
    ))

    bursts = []
    for start in starts:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
        burst = []
        for offset in range(burst_size):
            ok, frame = cap.read()
            if not ok:
                break
            burst.append((start + offset, frame))
        if burst:
            bursts.append(burst)
    cap.release()

    n_frames_total = sum(len(b) for b in bursts)
    if n_frames_total < total_frames:
        print(f"Warning: only decoded {n_frames_total}/{total_frames} frames "
              f"across {len(bursts)} bursts (some reads may have failed near the end of the file).",
              file=sys.stderr)
    return bursts


def compute_motion_scores(burst: list) -> dict:
    """
    Given one burst (list of (frame_idx, frame) consecutive frames), return
    {frame_idx: motion_score} where motion_score is derived from dense
    Farneback optical flow magnitude between neighboring frames, normalized
    by frame diagonal so it's roughly resolution-independent.

    This is a generic motion-magnitude signal, not action recognition --
    see module docstring. The absolute scale is empirical: print the
    average motion score across a run and use that to pick a sane
    --motion-threshold for the footage at hand rather than trusting a
    one-size-fits-all default.
    """

    if len(burst) < 2:
        return {burst[0][0]: 0.0} if burst else {}

    grays = [cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) for _, frame in burst]
    h, w = grays[0].shape[:2]
    diagonal = (h ** 2 + w ** 2) ** 0.5

    pair_scores = []
    for i in range(len(grays) - 1):
        flow = cv2.calcOpticalFlowFarneback(
            grays[i], grays[i + 1], None,
            pyr_scale=0.5, levels=3, winsize=15,
            iterations=3, poly_n=5, poly_sigma=1.2, flags=0,
        )
        magnitude = np.sqrt(flow[..., 0] ** 2 + flow[..., 1] ** 2)
        # Scaled by 1000 purely so printed values are human-readable
        # (raw pixel-flow-per-diagonal is a tiny fraction otherwise).
        pair_scores.append(float(magnitude.mean()) / diagonal * 1000)

    idxs = [idx for idx, _ in burst]
    scores = {}
    for i, idx in enumerate(idxs):
        if i == 0:
            scores[idx] = pair_scores[0]
        elif i == len(idxs) - 1:
            scores[idx] = pair_scores[-1]
        else:
            scores[idx] = (pair_scores[i - 1] + pair_scores[i]) / 2
    return scores


def run_model(weights_path: Path, bursts: list, conf_threshold: float = DEFAULT_CONF,
              motion_threshold: float | None = None, save_frames_dir: Path | None = None,
              model_name: str = "model"):
    """
    Runs the detector over every frame in every burst. If save_frames_dir
    is given, every frame with a raw detection gets saved as a JPEG with
    YOLO's own bounding boxes + class labels drawn on it (via
    ultralytics' Result.plot()), named so the frame index, confidence,
    and whether it also passed the motion gate are all visible in the
    filename without opening the image - useful for scanning a folder
    of hundreds of saved frames quickly to find real detections.

    Only frames with an actual raw detection are saved (not every frame
    analyzed) - saving all of them would produce mostly-identical empty
    frames and bury the interesting ones.
    """
    model = YOLO(str(weights_path))
    results = []
    if save_frames_dir is not None:
        Path(save_frames_dir).mkdir(parents=True, exist_ok=True)

    for burst in bursts:
        motion_scores = compute_motion_scores(burst)
        for idx, frame in burst:
            pred = model.predict(frame, conf=conf_threshold, verbose=False)[0]
            fight_confs = [
                float(box.conf[0])
                for box in pred.boxes
                if int(box.cls[0]) == FIGHT_CLASS_ID
            ]
            raw_detected = len(fight_confs) > 0
            motion_score = motion_scores.get(idx, 0.0)
            motion_gated = (
                raw_detected and motion_threshold is not None and motion_score >= motion_threshold
            )

            if save_frames_dir is not None and raw_detected:
                annotated = pred.plot()  # draws boxes + class labels + confidence
                max_conf = max(fight_confs)
                gate_tag = "_motiongated" if motion_gated else ("_nogate" if motion_threshold is not None else "")
                fname = f"{model_name}_frame{idx:06d}_conf{max_conf:.2f}{gate_tag}.jpg"
                cv2.imwrite(str(Path(save_frames_dir) / fname), annotated)

            results.append({
                "frame_idx": idx,
                "raw_detected": raw_detected,
                "max_conf": max(fight_confs) if fight_confs else 0.0,
                "motion_score": motion_score,
                "motion_gated_detected": motion_gated,
            })
    return results


def summarize(model_name: str, results: list[dict], motion_threshold: float | None) -> dict:
    n = len(results)
    n_raw = sum(r["raw_detected"] for r in results)
    avg_conf = sum(r["max_conf"] for r in results) / n if n else 0.0
    motion_scores = [r["motion_score"] for r in results]
    avg_motion = sum(motion_scores) / n if n else 0.0
    min_motion = min(motion_scores) if motion_scores else 0.0
    max_motion = max(motion_scores) if motion_scores else 0.0
    raw_verdict = "FIGHT" if n and (n_raw / n) >= 0.5 else "NO FIGHT"

    print(f"\n=== {model_name} ===")
    print(f"Frames analyzed:                 {n}")
    print(f"Raw YOLO fight frames:            {n_raw}/{n} ({100 * n_raw / n:.1f}%)  -> verdict: {raw_verdict}" if n else "Raw YOLO fight frames:            0")
    print(f"Average confidence:               {avg_conf:.3f}")
    print(f"Motion score  min/avg/max:        {min_motion:.3f} / {avg_motion:.3f} / {max_motion:.3f}  (unitless, use to pick --motion-threshold)")

    out = {
        "model": model_name,
        "n_frames": n,
        "n_raw": n_raw,
        "avg_conf": avg_conf,
        "avg_motion": avg_motion,
        "raw_verdict": raw_verdict,
    }

    if motion_threshold is not None:
        n_gated = sum(r["motion_gated_detected"] for r in results)
        gated_verdict = "FIGHT" if n and (n_gated / n) >= 0.5 else "NO FIGHT"
        print(f"Motion-gated fight frames:        {n_gated}/{n} ({100 * n_gated / n:.1f}%)  -> verdict: {gated_verdict} "
              f"(threshold={motion_threshold})")
        out["n_gated"] = n_gated
        out["gated_verdict"] = gated_verdict
    else:
        print("Motion-gated verdict:              (skipped - pass --motion-threshold to enable; "
              "see the min/avg/max above to pick one)")
        out["n_gated"] = None
        out["gated_verdict"] = None

    return out



import os as _os
def _assert_drive_mounted(path):
    if str(path).startswith("/content/drive") and not _os.path.ismount("/content/drive"):
        raise RuntimeError(
            f"Google Drive is not mounted at /content/drive (tried to use {path}). "
            "Run the Drive-mount cell first, or pass --weights-dir pointing "
            "somewhere local to avoid persisting these weights."
        )

def ensure_weights(weights_dir: Path, download: bool) -> tuple[Path, Path]:
    _assert_drive_mounted(weights_dir)
    nano_path = weights_dir / "Yolo_nano_weights.pt"
    small_path = weights_dir / "yolo_small_weights.pt"

    if nano_path.exists() and small_path.exists():
        return nano_path, small_path

    if not download:
        missing = [p.name for p in (nano_path, small_path) if not p.exists()]
        sys.exit(
            f"Missing weight file(s): {missing}. Download them from "
            f"https://huggingface.co/Musawer14/fight_detection_yolov8/tree/main "
            f"and place them in {weights_dir}, or re-run with --download."
        )

    from huggingface_hub import hf_hub_download
    repo_id = "Musawer14/fight_detection_yolov8"
    if not nano_path.exists():
        print("Downloading Yolo_nano_weights.pt ...")
        downloaded = hf_hub_download(repo_id=repo_id, filename="Yolo_nano_weights.pt", local_dir=weights_dir)
        nano_path = Path(downloaded)
    if not small_path.exists():
        print("Downloading yolo_small_weights.pt ...")
        downloaded = hf_hub_download(repo_id=repo_id, filename="yolo_small_weights.pt", local_dir=weights_dir)
        small_path = Path(downloaded)

    return nano_path, small_path


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--video", required=True, help="Path to the video file to analyze.")
    parser.add_argument("--total-frames", type=int, default=30,
                         help="Total frames to sample, must be a multiple of --burst-size (min 30 per the task requirement).")
    parser.add_argument("--burst-size", type=int, default=5,
                         help="Consecutive frames per burst. Motion is only computed within a burst.")
    parser.add_argument("--motion-threshold", type=float, default=None,
                         help="Enable motion-gated verdicts: a frame only counts as 'fight' if YOLO confidence AND "
                              "this motion score are both met. No universal default exists - run once without this "
                              "flag, read the printed min/avg/max motion score, then pick a value from that range.")
    parser.add_argument("--weights-dir", default=".", help="Directory containing/receiving the .pt weight files.")
    parser.add_argument("--conf", type=float, default=DEFAULT_CONF, help="Confidence threshold for a positive detection.")
    parser.add_argument("--download", action="store_true", help="Auto-download missing weights from Hugging Face.")
    parser.add_argument("--skip-small", action="store_true", help="Only run the nano (Safe-scanned) checkpoint; skips loading yolo_small_weights.pt entirely.")
    parser.add_argument("--save-frames-dir", default=None,
                         help="If given, save every frame with a raw detection as a JPEG with bounding boxes "
                              "drawn on it (via YOLO's own Result.plot()) into this directory. Only detected "
                              "frames are saved, not every frame analyzed.")
    args = parser.parse_args()

    if args.total_frames < 30:
        sys.exit("--total-frames must be at least 30 per the task requirement.")
    if args.total_frames % args.burst_size != 0:
        sys.exit(f"--total-frames ({args.total_frames}) must be a multiple of --burst-size ({args.burst_size}).")

    weights_dir = Path(args.weights_dir)
    nano_path, small_path = ensure_weights(weights_dir, args.download)

    bursts = sample_frame_bursts(Path(args.video), args.total_frames, args.burst_size)
    n_frames = sum(len(b) for b in bursts)
    print(f"Extracted {n_frames} frames across {len(bursts)} bursts of {args.burst_size} "
          f"consecutive frames each from {args.video}")

    summaries = []

    save_dir = Path(args.save_frames_dir) if args.save_frames_dir else None

    print("\nRunning YOLOv8-nano (Safe-scanned checkpoint)...")
    summaries.append(summarize(
        "YOLOv8-nano",
        run_model(nano_path, bursts, args.conf, args.motion_threshold,
                  save_frames_dir=save_dir, model_name="nano"),
        args.motion_threshold
    ))

    if not args.skip_small:
        print("\nChecking yolo_small_weights.pt pickle stream before loading (Unsafe-scanned checkpoint)...")
        flagged = inspect_pickle_safety(small_path)
        if flagged:
            print("Suspicious references found - refusing to load automatically:")
            for item in flagged:
                print(f"  - {item}")
            print("\nSkipping YOLOv8-small. Re-run with --skip-small to suppress this branch entirely,")
            print("or manually review the checkpoint before forcing a load.")
        else:
            print("No suspicious references found in the pickle stream (heuristic check only - see script docstring).")
            print("Running YOLOv8-small...")
            summaries.append(summarize(
                "YOLOv8-small",
                run_model(small_path, bursts, args.conf, args.motion_threshold,
                          save_frames_dir=save_dir, model_name="small"),
                args.motion_threshold
            ))

    if save_dir is not None:
        n_saved = len(list(save_dir.glob("*.jpg")))
        print(f"\nSaved {n_saved} annotated detection frames to {save_dir}")

    print("\n=== Comparison ===")
    for s in summaries:
        gated_str = f", gated {s['n_gated']}/{s['n_frames']} ({s['gated_verdict']})" if s["gated_verdict"] else ""
        print(f"{s['model']:>14}: raw {s['n_raw']}/{s['n_frames']} ({s['raw_verdict']}){gated_str}, "
              f"avg conf {s['avg_conf']:.3f}, avg motion {s['avg_motion']:.3f}")


if __name__ == "__main__":
    main()

### Running it

Point `--video` at any video file (a UCLA training video, a UBI-Fights clip,
anything). `--download` fetches both YOLO checkpoints from Hugging Face on
first run. `--save-frames-dir` is what draws and saves the bounding-box
frames - only frames with an actual detection get saved, not every frame
analyzed, avoiding a scroll through hundreds of empty images.

Replace `/content/data/your_video.mp4` with a real video path before running.

In [ ]:
!python extract_violent_frames.py \
    --video /content/data/your_video.mp4 \
    --download \
    --weights-dir /content/drive/MyDrive/violence_frame_weights \
    --save-frames-dir /content/data/violent_frames \
    --total-frames 30


### Preview a few of the saved detection frames inline

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

saved = sorted(Path("/content/data/violent_frames").glob("*.jpg"))
print(f"{len(saved)} detection frames saved.")

n_preview = min(6, len(saved))
if n_preview:
    fig, axes = plt.subplots(1, n_preview, figsize=(4 * n_preview, 4))
    if n_preview == 1:
        axes = [axes]
    for ax, path in zip(axes, saved[:n_preview]):
        ax.imshow(mpimg.imread(path))
        ax.set_title(path.name, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
